# CORA 50M — Fast Training (T4, 15-20 min)

Pipeline completo: Encoder(Mamba) → Crystallizer → CRE → Decoder
- Config primaria: **hidden_dim=256, 8 layers, CRE=3 iter** (~37M params)
- Auto-fallback:   **hidden_dim=128, 4 layers, CRE=3 iter** ( ~5M params) si >25 min
- 2000 steps · batch=1 · cosine LR · checkpoints en Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_DIR = '/content/drive/MyDrive/cora_50m_fast'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results',     exist_ok=True)
print(f'Drive montado. Directorio: {DRIVE_DIR}')

In [ ]:
import os, pathlib, base64, sys

ROOT = pathlib.Path('/content/aion_c')
for pkg in ['core', 'synth', 'encoder', 'crystallizer', 'cre', 'decoder', 'router', 'experiments']:
    (ROOT / pkg).mkdir(parents=True, exist_ok=True)

files_b64 = {
    'core/__init__.py': (
        'IiIiCkFJT04tQyBjb3JlIOKAlCB0aXBvcyB5IGRhdGFjbGFzc2VzIGNvbXBhcnRpZG9zIHBvciB0'
        'b2RvcyBsb3MgbcOzZHVsb3MgZGUgQ09SQS4KIiIiCgpmcm9tIC5ncmFwaCBpbXBvcnQgKAogICAg'
        'Q2F1c2FsRWRnZSwKICAgIENhdXNhbEdyYXBoLAogICAgQ2F1c2FsTm9kZSwKICAgIENhdXNhbFJl'
        'bGF0aW9uLAogICAgTm9kZVR5cGUsCiAgICBDQVVTQUxfUkVMQVRJT05TLAogICAgQ0FVU0FMX1JF'
        'TEFUSU9OU19MSVNULAogICAgQ09OVFJBRElDVElPTl9QQUlSUywKICAgIElOSElCSVRPUllfUkVM'
        'QVRJT05TLAogICAgTk9ERV9UWVBFUywKICAgIFBPU0lUSVZFX1JFTEFUSU9OUywKICAgIFNUUlVD'
        'VFVSQUxfUkVMQVRJT05TLAogICAgU1lNTUVUUklDX1JFTEFUSU9OUywKICAgIFRFTVBPUkFMX1JF'
        'TEFUSU9OUywKKQoKX19hbGxfXyA9IFsKICAgICJDYXVzYWxFZGdlIiwKICAgICJDYXVzYWxHcmFw'
        'aCIsCiAgICAiQ2F1c2FsTm9kZSIsCiAgICAiQ2F1c2FsUmVsYXRpb24iLAogICAgIk5vZGVUeXBl'
        'IiwKICAgICJDQVVTQUxfUkVMQVRJT05TIiwKICAgICJDQVVTQUxfUkVMQVRJT05TX0xJU1QiLAog'
        'ICAgIkNPTlRSQURJQ1RJT05fUEFJUlMiLAogICAgIklOSElCSVRPUllfUkVMQVRJT05TIiwKICAg'
        'ICJOT0RFX1RZUEVTIiwKICAgICJQT1NJVElWRV9SRUxBVElPTlMiLAogICAgIlNUUlVDVFVSQUxf'
        'UkVMQVRJT05TIiwKICAgICJTWU1NRVRSSUNfUkVMQVRJT05TIiwKICAgICJURU1QT1JBTF9SRUxB'
        'VElPTlMiLApdCg=='
    ),
    'core/graph.py': (
        '...'
        'IGl0ZXIoc2VsZi5lZGdlcykK'
    ),
    'synth/__init__.py': (
        'IiIiCkFJT04tQyBzeW50aCDigJQgR2VuZXJhZG9yZXMgZGUgZGF0b3Mgc2ludMOpdGljb3MgcGFy'
        'YSBlbnRyZW5hbWllbnRvLgoiIiIKCmZyb20gLmNhdXNhbF9ncmFwaF9nZW4gaW1wb3J0ICgKICAg'
        'IEFuc3dlclR5cGUsCiAgICBDYXVzYWxFeGFtcGxlLAogICAgQ2F1c2FsR3JhcGhHZW5lcmF0b3Is'
        'CiAgICBWZXJpZmljYXRpb25SZXN1bHQsCiAgICB2ZXJpZnlfZXhhbXBsZSwKKQoKX19hbGxfXyA9'
        'IFsKICAgICJBbnN3ZXJUeXBlIiwKICAgICJDYXVzYWxFeGFtcGxlIiwKICAgICJDYXVzYWxHcmFw'
        'aEdlbmVyYXRvciIsCiAgICAiVmVyaWZpY2F0aW9uUmVzdWx0IiwKICAgICJ2ZXJpZnlfZXhhbXBs'
        'ZSIsCl0K'
    ),
    'synth/causal_graph_gen.py': (
        '...'
        'KQo='
    ),
    'encoder/__init__.py': (
        'IiIiCkFJT04tQyBlbmNvZGVyIOKAlCBTdHJlYW1FbmNvZGVyIGJhc2FkbyBlbiBNYW1iYS1zdHls'
        'ZSBTU00uCgpDb252aWVydGUgdG9rZW4gSURzIGVuIGNvbmNlcHQgdmVjdG9ycyBjb24gY29tcGxl'
        'amlkYWQgTyhMKSwKZXZpdGFuZG8gbGEgTyhMwrIpIGRlIGF0dGVudGlvbi4gUGFzbyBwcmV2aW8g'
        'YWwgR3JhcGhDb25zdHJ1Y3Rvci4KIiIiCgpmcm9tIC5tYW1iYV9sYXllciBpbXBvcnQgKAogICAg'
        'R2F0ZWRGRk4sCiAgICBNYW1iYUxheWVyLAogICAgUk1TTm9ybSwKICAgIFNlbGVjdGl2ZVNTTSwK'
        'ICAgIFN0cmVhbUVuY29kZXJDb25maWcsCikKZnJvbSAubW9kZWwgaW1wb3J0IFN0cmVhbUVuY29k'
        'ZXIKCl9fYWxsX18gPSBbCiAgICAiR2F0ZWRGRk4iLAogICAgIk1hbWJhTGF5ZXIiLAogICAgIlJN'
        'U05vcm0iLAogICAgIlNlbGVjdGl2ZVNTTSIsCiAgICAiU3RyZWFtRW5jb2RlciIsCiAgICAiU3Ry'
        'ZWFtRW5jb2RlckNvbmZpZyIsCl0K'
    ),
    'encoder/mamba_layer.py': (
        '...'
        'cm4geCwgc3RhdGVzCg=='
    ),
    'encoder/model.py': (
        'IiIiCmVuY29kZXIvbW9kZWwucHkg4oCUIFN0cmVhbUVuY29kZXI6IHRva2VucyDihpIgY29uY2Vw'
        'dCB2ZWN0b3JzCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09PT09CgpFbCBTdHJlYW1FbmNvZGVyIGVzIGVsIHByaW1lciBtw7NkdWxvIGRlbCBw'
        'aXBlbGluZSBBSU9OLUMgQ0VOOgoKICB0b2tlbl9pZHMgW0IsIExdCiAgICAgICDihpMKICB0b2tl'
        'bl9lbWJlZGRpbmcgW0IsIEwsIERdCiAgICAgICDihpMKICBNYW1iYUxheWVyIMOXIG5fbGF5ZXJz'
        'CiAgICAgICDihpMKICBSTVNOb3JtCiAgICAgICDihpMKICBjb25jZXB0X3Byb2plY3RvciBbQiwg'
        'TCwgY29uY2VwdF9kaW1dCiAgICAgICDihpMKICBjb25jZXB0X3ZlY3RvcnMg4oaSIEdyYXBoQ29u'
        'c3RydWN0b3IKClByb3BpZWRhZCBjbGF2ZTogbWVtb3JpYSBPKEwpLCBubyBPKEzCsikuCkVsIHNj'
        'YW4gc2VjdWVuY2lhbCBkZWwgU1NNIG1hbnRpZW5lIHVuIGVzdGFkbyBoW0IsIERfaW5uZXIsIE5d'
        'Cih0YW1hw7FvIGNvbnN0YW50ZSkgbWllbnRyYXMgcHJvY2VzYSBjYWRhIHRva2VuLiBMYSBpbmZv'
        'cm1hY2nDs24KZGVsIHBhc2FkbyBxdWVkYSBjb21wcmltaWRhIGVuIGVzZSB2ZWN0b3IgZGUgZXN0'
        'YWRvLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gdHlwaW5n'
        'IGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUKCmltcG9ydCB0b3JjaAppbXBvcnQg'
        'dG9yY2gubm4gYXMgbm4KCmZyb20gLm1hbWJhX2xheWVyIGltcG9ydCBHYXRlZEZGTiwgTWFtYmFM'
        'YXllciwgUk1TTm9ybSwgU3RyZWFtRW5jb2RlckNvbmZpZwoKCiMg4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiMgU1RSRUFNIEVOQ09E'
        'RVIKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIAKCmNsYXNzIFN0cmVhbUVuY29kZXIobm4uTW9kdWxlKToKICAgICIiIgogICAgQ29u'
        'dmllcnRlIHRva2VuIElEcyBlbiBjb25jZXB0IHZlY3RvcnMgdXNhbmRvIE1hbWJhLXN0eWxlIFNT'
        'TS4KCiAgICBFbCBjb25jZXB0byBkZSAiZXNwYWNpbyBjb25jZXB0dWFsIiAoY29uY2VwdF9kaW0g'
        'PCBoaWRkZW5fZGltKToKICAgICAgInRoZSBxdWljayBicm93biBmb3giID0gNCB0b2tlbnMg4oaS'
        'IDItMyBjb25jZXB0b3MuCiAgICAgIExhIGNvbXByZXNpw7NuIGVsaW1pbmEgaW5mb3JtYWNpw7Nu'
        'IGzDqXhpY2Egc3VwZXJmaWNpYWwsCiAgICAgIGRlamFuZG8gc29sbyBsYSBzZW3DoW50aWNhIHJl'
        'bGV2YW50ZSBwYXJhIGVsIENFQy4KICAgICAgRWwgQ0VDIG9wZXJhIGVuIGNvbmNlcHRfZGltPTEy'
        'OGQsIG5vIGVuIGhpZGRlbl9kaW09MjU2ZC4KICAgICAg4oaSIDR4IG1lbm9zIGNvbXB1dGUgcG9y'
        'IG9wZXJhY2nDs24gZW4gZWwgcmF6b25hZG9yLgoKICAgIENvbmZpZ3VyYWNpw7NuIHRpbnkgKHRl'
        'c3RpbmcpOgogICAgICB2b2NhYl9zaXplPTMyMDAwLCBoaWRkZW5fZGltPTI1Niwgbl9sYXllcnM9'
        'NCwKICAgICAgc3RhdGVfZGltPTE2LCBjb25jZXB0X2RpbT0xMjgKICAgICAgUGFyw6FtZXRyb3Mg'
        'dG90YWxlczogfjEzTQoKICAgIFVzbzoKICAgICAgICBjb25maWcgID0gU3RyZWFtRW5jb2RlckNv'
        'bmZpZygpCiAgICAgICAgZW5jb2RlciA9IFN0cmVhbUVuY29kZXIoY29uZmlnKQogICAgICAgIGlk'
        'cyAgICAgPSB0b3JjaC5yYW5kaW50KDAsIGNvbmZpZy52b2NhYl9zaXplLCAoMiwgNTEyKSkKICAg'
        'ICAgICB2ZWNzICAgID0gZW5jb2RlcihpZHMpICAgIyBbMiwgNTEyLCAxMjhdCiAgICAiIiIKCiAg'
        'ICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBTdHJlYW1FbmNvZGVyQ29uZmlnKSAtPiBOb25l'
        'OgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuY29uZmlnID0gY29uZmln'
        'CgogICAgICAgICMgRW1iZWRkaW5nIGRlIHRva2VucwogICAgICAgIHNlbGYudG9rZW5fZW1iZWRk'
        'aW5nID0gbm4uRW1iZWRkaW5nKGNvbmZpZy52b2NhYl9zaXplLCBjb25maWcuaGlkZGVuX2RpbSkK'
        'CiAgICAgICAgIyBTdGFjayBkZSBjYXBhcyBNYW1iYQogICAgICAgIHNlbGYubGF5ZXJzID0gbm4u'
        'TW9kdWxlTGlzdChbCiAgICAgICAgICAgIE1hbWJhTGF5ZXIoY29uZmlnKSBmb3IgXyBpbiByYW5n'
        'ZShjb25maWcubl9sYXllcnMpCiAgICAgICAgXSkKCiAgICAgICAgIyBOb3JtYWxpemFjacOzbiBm'
        'aW5hbAogICAgICAgIHNlbGYubm9ybSA9IFJNU05vcm0oY29uZmlnLmhpZGRlbl9kaW0sIGVwcz1j'
        'b25maWcucm1zX2VwcykKCiAgICAgICAgIyBQcm95ZWNjacOzbiBhbCBlc3BhY2lvIGNvbmNlcHR1'
        'YWwgKGhpZGRlbl9kaW0g4oaSIGNvbmNlcHRfZGltKQogICAgICAgICMgU2luIGJpYXMg4oCUIGxh'
        'IGluZm9ybWFjacOzbiB2aWVuZSBkZWwgZW1iZWRkaW5nICsgU1NNCiAgICAgICAgc2VsZi5jb25j'
        'ZXB0X3Byb2plY3RvciA9IG5uLkxpbmVhcigKICAgICAgICAgICAgY29uZmlnLmhpZGRlbl9kaW0s'
        'IGNvbmZpZy5jb25jZXB0X2RpbSwgYmlhcz1GYWxzZQogICAgICAgICkKCiAgICAgICAgc2VsZi5f'
        'aW5pdF93ZWlnaHRzKCkKCiAgICAjIOKUgOKUgCBJbmljaWFsaXphY2nDs24g4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIF9pbml0X3dl'
        'aWdodHMoc2VsZikgLT4gTm9uZToKICAgICAgICAiIiIKICAgICAgICBJbmljaWFsaXphY2nDs24g'
        'ZXN0w6FuZGFyIHBhcmEgTExNczoKICAgICAgICAtIEVtYmVkZGluZzogbm9ybWFsKHN0ZD0wLjAy'
        'KQogICAgICAgIC0gY29uY2VwdF9wcm9qZWN0b3I6IG5vcm1hbChzdGQ9MC4wMiAvIHNxcnQoMiAq'
        'IG5fbGF5ZXJzKSkKICAgICAgICAgIEVsIGZhY3RvciAxL3NxcnQoMkwpIHJlZHVjZSBsYSB2YXJp'
        'YW56YSBkZSBsb3MgcmVzaWR1YWxzCiAgICAgICAgICBhbCBhY3VtdWxhciBMIGNhcGFzIChzaW1p'
        'bGFyIGEgR1BULTIpLgogICAgICAgICIiIgogICAgICAgIHN0ZF9lbWIgID0gMC4wMgogICAgICAg'
        'IHN0ZF9wcm9qID0gMC4wMiAvICgyICogc2VsZi5jb25maWcubl9sYXllcnMpICoqIDAuNQoKICAg'
        'ICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi50b2tlbl9lbWJlZGRpbmcud2VpZ2h0LCBzdGQ9c3Rk'
        'X2VtYikKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5jb25jZXB0X3Byb2plY3Rvci53ZWln'
        'aHQsIHN0ZD1zdGRfcHJvaikKCiAgICAjIOKUgOKUgCBGb3J3YXJkIHByaW5jaXBhbCDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCiAgICBkZWYgZm9yd2FyZChzZWxm'
        'LCB0b2tlbl9pZHM6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIgog'
        'ICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHRva2VuX2lkczogW0IsIExdIOKAlCBsb25nIHRlbnNv'
        'ciBkZSBJRHMgZGUgdG9rZW5zCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIGNvbmNlcHRf'
        'dmVjdG9yczogW0IsIEwsIGNvbmNlcHRfZGltXQogICAgICAgICAgICAgIENhZGEgcG9zaWNpw7Nu'
        'IGwgY29udGllbmUgZWwgY29uY2VwdG8gZXh0cmHDrWRvIGRlbCBjb250ZXh0bwogICAgICAgICAg'
        'ICAgIFswLi5sXSAoZ3JhY2lhcyBhbCBzY2FuIGNhdXNhbCBkZWwgU1NNKS4KICAgICAgICAiIiIK'
        'ICAgICAgICB4ID0gc2VsZi50b2tlbl9lbWJlZGRpbmcodG9rZW5faWRzKSAgIyBbQiwgTCwgRF0K'
        'CiAgICAgICAgZm9yIGxheWVyIGluIHNlbGYubGF5ZXJzOgogICAgICAgICAgICB4LCBfID0gbGF5'
        'ZXIoeCkKCiAgICAgICAgeCA9IHNlbGYubm9ybSh4KQogICAgICAgIHJldHVybiBzZWxmLmNvbmNl'
        'cHRfcHJvamVjdG9yKHgpICAgICAjIFtCLCBMLCBjb25jZXB0X2RpbV0KCiAgICAjIOKUgOKUgCBG'
        'b3J3YXJkIGNvbiBpbnNwZWNjacOzbiBpbnRlcm5hIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBmb3J3YXJkX3dpdGhf'
        'c3RhdGVzKAogICAgICAgIHNlbGYsCiAgICAgICAgdG9rZW5faWRzOiB0b3JjaC5UZW5zb3IsCiAg'
        'ICApIC0+IFR1cGxlW3RvcmNoLlRlbnNvciwgTGlzdFt0b3JjaC5UZW5zb3JdXToKICAgICAgICAi'
        'IiIKICAgICAgICBJZ3VhbCBxdWUgZm9yd2FyZCgpIHBlcm8gcmV0b3JuYSB0YW1iacOpbiBsYSB0'
        'cmF5ZWN0b3JpYSBkZSBlc3RhZG9zIFNTTS4KCiAgICAgICAgVXNvOgogICAgICAgICAgICBjb25j'
        'ZXB0cywgc3RhdGVzID0gZW5jb2Rlci5mb3J3YXJkX3dpdGhfc3RhdGVzKGlkcykKICAgICAgICAg'
        'ICAgIyBzdGF0ZXNbaV06IFtCLCBMLCBEX2lubmVyLCBOXSDigJQgZXN0YWRvcyBkZSBsYSBjYXBh'
        'IGkKICAgICAgICAgICAgIyBzdGF0ZXNbaV1bOiwgdCwgOiwgOl0g4oCUIGVzdGFkbyBlbiBlbCB0'
        'aW1lc3RlcCB0LCBjYXBhIGkKCiAgICAgICAgVXRpbCBwYXJhOgogICAgICAgICAgICAtIFZlcmlm'
        'aWNhciBxdWUgZWwgZXN0YWRvIFNTTSBjYW1iaWEgZW50cmUgdGltZXN0ZXBzCiAgICAgICAgICAg'
        'IC0gRGVidWdnaW5nIGRlbCBzY2FuCiAgICAgICAgICAgIC0gVmlzdWFsaXphY2nDs24gZGUgcXXD'
        'qSBpbmZvcm1hY2nDs24gcmV0aWVuZSBlbCBlc3RhZG8KICAgICAgICAiIiIKICAgICAgICB4ID0g'
        'c2VsZi50b2tlbl9lbWJlZGRpbmcodG9rZW5faWRzKQogICAgICAgIGFsbF9zdGF0ZXM6IExpc3Rb'
        'dG9yY2guVGVuc29yXSA9IFtdCgogICAgICAgIGZvciBsYXllciBpbiBzZWxmLmxheWVyczoKICAg'
        'ICAgICAgICAgeCwgc3RhdGVzID0gbGF5ZXIoeCwgcmV0dXJuX3N0YXRlcz1UcnVlKQogICAgICAg'
        'ICAgICBhbGxfc3RhdGVzLmFwcGVuZChzdGF0ZXMpICAgIyBzdGF0ZXM6IFtCLCBMLCBEX2lubmVy'
        'LCBOXQoKICAgICAgICB4ID0gc2VsZi5ub3JtKHgpCiAgICAgICAgY29uY2VwdHMgPSBzZWxmLmNv'
        'bmNlcHRfcHJvamVjdG9yKHgpCiAgICAgICAgcmV0dXJuIGNvbmNlcHRzLCBhbGxfc3RhdGVzCgog'
        'ICAgZGVmIGdldF9zc21fQV9iYXJfbnVtZWwoc2VsZiwgdG9rZW5faWRzOiB0b3JjaC5UZW5zb3Ip'
        'IC0+IGludDoKICAgICAgICAiIiIKICAgICAgICBFamVjdXRhIHVuIGZvcndhcmQgeSByZXRvcm5h'
        'IGVsIG7Dum1lcm8gZGUgZWxlbWVudG9zIGRlbCB0ZW5zb3IgQV9iYXIKICAgICAgICBjb21wdXRh'
        'ZG8gZW4gbGEgcHJpbWVyYSBjYXBhIFNTTS4KCiAgICAgICAgVXNhZG8gZW4gdGVzdHMgZGUgZXNj'
        'YWxhZG8gZGUgbWVtb3JpYToKICAgICAgICAgIEFfYmFyLm51bWVsKCkgPSBCIMOXIEwgw5cgRF9p'
        'bm5lciDDlyBOICDihpAgbGluZWFsIGVuIEwsIE5PIGN1YWRyw6F0aWNvCgogICAgICAgIERldnVl'
        'bHZlIHNpZW1wcmUgZWwgdmFsb3IgZGUgbGEgUFJJTUVSQSBjYXBhIChsYXllcnNbMF0uc3NtKS4K'
        'ICAgICAgICAiIiIKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgc2Vs'
        'Zi5mb3J3YXJkKHRva2VuX2lkcykKICAgICAgICByZXR1cm4gc2VsZi5sYXllcnNbMF0uc3NtLl9s'
        'YXN0X0FfYmFyX251bWVsCgogICAgIyDilIDilIAgVXRpbGlkYWRlcyDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCiAgICBk'
        'ZWYgY291bnRfcGFyYW1ldGVycyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgIiIiTsO6bWVybyB0b3Rh'
        'bCBkZSBwYXLDoW1ldHJvcyBlbnRyZW5hYmxlcy4iIiIKICAgICAgICByZXR1cm4gc3VtKHAubnVt'
        'ZWwoKSBmb3IgcCBpbiBzZWxmLnBhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWQpCgogICAg'
        'ZGVmIHBhcmFtZXRlcl9icmVha2Rvd24oc2VsZikgLT4gRGljdFtzdHIsIGludF06CiAgICAgICAg'
        'IiIiRGVzZ2xvc2UgZGUgcGFyw6FtZXRyb3MgcG9yIG3Ds2R1bG8uIiIiCiAgICAgICAgcmV0dXJu'
        'IHsKICAgICAgICAgICAgInRva2VuX2VtYmVkZGluZyI6IHNlbGYudG9rZW5fZW1iZWRkaW5nLndl'
        'aWdodC5udW1lbCgpLAogICAgICAgICAgICAibGF5ZXJzX3NzbSI6ICAgICAgc3VtKAogICAgICAg'
        'ICAgICAgICAgcC5udW1lbCgpCiAgICAgICAgICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXll'
        'cnMKICAgICAgICAgICAgICAgIGZvciBwIGluIGxheWVyLnNzbS5wYXJhbWV0ZXJzKCkKICAgICAg'
        'ICAgICAgKSwKICAgICAgICAgICAgImxheWVyc19mZm4iOiAgICAgIHN1bSgKICAgICAgICAgICAg'
        'ICAgIHAubnVtZWwoKQogICAgICAgICAgICAgICAgZm9yIGxheWVyIGluIHNlbGYubGF5ZXJzCiAg'
        'ICAgICAgICAgICAgICBmb3IgcCBpbiBsYXllci5mZm4ucGFyYW1ldGVycygpCiAgICAgICAgICAg'
        'ICksCiAgICAgICAgICAgICJsYXllcnNfbm9ybXMiOiAgICBzdW0oCiAgICAgICAgICAgICAgICBw'
        'Lm51bWVsKCkKICAgICAgICAgICAgICAgIGZvciBsYXllciBpbiBzZWxmLmxheWVycwogICAgICAg'
        'ICAgICAgICAgZm9yIHAgaW4gbGlzdChsYXllci5ub3JtMS5wYXJhbWV0ZXJzKCkpICsgbGlzdChs'
        'YXllci5ub3JtMi5wYXJhbWV0ZXJzKCkpCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJjb25j'
        'ZXB0X3Byb2plY3RvciI6IHNlbGYuY29uY2VwdF9wcm9qZWN0b3Iud2VpZ2h0Lm51bWVsKCksCiAg'
        'ICAgICAgICAgICJ0b3RhbCI6IHNlbGYuY291bnRfcGFyYW1ldGVycygpLAogICAgICAgIH0K'
    ),
    'crystallizer/__init__.py': (
        'IiIiCkFJT04tQyBjcnlzdGFsbGl6ZXIg4oCUIEdyYXBoQ3J5c3RhbGxpemVyOiBjb25jZXB0IHZl'
        'Y3RvcnMg4oaSIENhdXNhbEdyYXBoLgoKQ29udmllcnRlIGNvbmNlcHQgdmVjdG9ycyBbQiwgTCwg'
        'RF0gKGRlbCBTdHJlYW1FbmNvZGVyKSBlbiBncmFmb3MgY2F1c2FsZXMKZXN0cnVjdHVyYWRvcywg'
        'dXNhbmRvIGRldGVjY2nDs24gZGUgbm9kb3MsIHBvb2xpbmcgcG9yIGNyb3NzLWF0dGVudGlvbgp5'
        'IHB1bnR1YWNpw7NuIGRlIHJlbGFjaW9uZXMgYXNpbcOpdHJpY2FzLgoiIiIKCmZyb20gLmNvbmZp'
        'ZyBpbXBvcnQgQ3J5c3RhbGxpemVyQ29uZmlnCmZyb20gLm1vZGVsIGltcG9ydCBDcnlzdGFsbGl6'
        'ZXJPdXRwdXQsIEdyYXBoQ3J5c3RhbGxpemVyCmZyb20gLm5vZGVfZGV0ZWN0b3IgaW1wb3J0IE5v'
        'ZGVEZXRlY3Rvcgpmcm9tIC5wb29sZXIgaW1wb3J0IENyb3NzQXR0ZW50aW9uUG9vbGVyCmZyb20g'
        'LnJlbGF0aW9uX3Njb3JlciBpbXBvcnQgQXN5bW1ldHJpY1JlbGF0aW9uU2NvcmVyCgpfX2FsbF9f'
        'ID0gWwogICAgIkFzeW1tZXRyaWNSZWxhdGlvblNjb3JlciIsCiAgICAiQ3Jvc3NBdHRlbnRpb25Q'
        'b29sZXIiLAogICAgIkNyeXN0YWxsaXplckNvbmZpZyIsCiAgICAiQ3J5c3RhbGxpemVyT3V0cHV0'
        'IiwKICAgICJHcmFwaENyeXN0YWxsaXplciIsCiAgICAiTm9kZURldGVjdG9yIiwKXQo='
    ),
    'crystallizer/config.py': (
        'IiIiCmNyeXN0YWxsaXplci9jb25maWcucHkg4oCUIENvbmZpZ3VyYWNpw7NuIGRlbCBHcmFwaENy'
        'eXN0YWxsaXplcgo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09PT09PT09CiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoK'
        'ZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAoKCkBkYXRhY2xhc3MKY2xh'
        'c3MgQ3J5c3RhbGxpemVyQ29uZmlnOgogICAgIiIiCiAgICBIaXBlcnBhcsOhbWV0cm9zIGRlbCBH'
        'cmFwaENyeXN0YWxsaXplci4KCiAgICBEaXNlw7FhZG8gcGFyYSBjb2luY2lkaXIgY29uIGVsIFN0'
        'cmVhbUVuY29kZXIgdXBzdHJlYW06CiAgICAgICAgaGlkZGVuX2RpbSA9IFN0cmVhbUVuY29kZXJD'
        'b25maWcuY29uY2VwdF9kaW0gPSAxMjggIChvIDI1NiBlbiB0aW55KQogICAgICAgIG5fcmVsYXRp'
        'b25fdHlwZXMgPSBsZW4oQ0FVU0FMX1JFTEFUSU9OUykgPSAxNgogICAgICAgIG5fbm9kZV90eXBl'
        'cyAgICAgPSBsZW4oTm9kZVR5cGUpICAgICAgICAgPSA3CgogICAgQ29uZmlndXJhY2nDs24gdGlu'
        'eSBwYXJhIHRlc3Rpbmc6CiAgICAgICAgaGlkZGVuX2RpbT0yNTYsIG1heF9ub2Rlcz0zMgoKICAg'
        'IFVtYnJhbGVzOgogICAgICAgIG5vZGVfdGhyZXNob2xkICDigJQgc2lnbW9pZCBzY29yZSBtw61u'
        'aW1vIHBhcmEgY29uc2lkZXJhciB1biBub2RvCiAgICAgICAgZWRnZV90aHJlc2hvbGQgIOKAlCBz'
        'aWdtb2lkIHNjb3JlIG3DrW5pbW8gcGFyYSBhZ3JlZ2FyIHVuYSBhcmlzdGEKICAgICAgICBWYWxv'
        'cmVzIGJham9zICgwLjMpIGdhcmFudGl6YW4gZ3JhZm9zIG5vIHZhY8Otb3MgZW4gdGVzdHMgY29u'
        'IHBlc29zIHJhbmRvbS4KICAgICIiIgoKICAgICMgRGltZW5zaW9uZXMKICAgIGhpZGRlbl9kaW06'
        'IGludCAgID0gMjU2ICAgIyA9IGNvbmNlcHRfZGltIGRlbCBTdHJlYW1FbmNvZGVyCiAgICBtYXhf'
        'bm9kZXM6IGludCAgICA9IDMyICAgICMgbcOheGltbyBkZSBub2RvcyBwb3IgZ3JhZm8gb3V0cHV0'
        'CgogICAgIyBWb2NhYnVsYXJpb3MgKGRlYmVuIGNvaW5jaWRpciBjb24gY29yZS9ncmFwaC5weSkK'
        'ICAgIG5fcmVsYXRpb25fdHlwZXM6IGludCA9IDE2ICAgIyBsZW4oQ0FVU0FMX1JFTEFUSU9OUykK'
        'ICAgIG5fbm9kZV90eXBlczogICAgIGludCA9IDcgICAgIyBsZW4oTm9kZVR5cGUpCgogICAgIyBV'
        'bWJyYWxlcyBkZSBkZXRlY2Npw7NuCiAgICBub2RlX3RocmVzaG9sZDogZmxvYXQgPSAwLjMgICMg'
        'cHJvYmFiaWxpZGFkIG3DrW5pbWEgcGFyYSBzZXIgbm9kbwogICAgZWRnZV90aHJlc2hvbGQ6IGZs'
        'b2F0ID0gMC4zICAjIHNpZ21vaWQgc2NvcmUgbcOtbmltbyBwYXJhIHNlciBhcmlzdGEKCiAgICAj'
        'IEFycXVpdGVjdHVyYSBkZWwgcG9vbGVyCiAgICBwb29sZXJfaGVhZHM6IGludCA9IDQgICAgICAg'
        'ICMgY2FiZXphcyBlbiBDcm9zc0F0dGVudGlvblBvb2xlcgoKICAgICMgQXJxdWl0ZWN0dXJhIGRl'
        'bCByZWxhdGlvbiBzY29yZXIKICAgIHJlbGF0aW9uX2hpZGRlbl9kaW06IGludCA9IDY0ICAjIGRp'
        'bWVuc2nDs24gcG9yIHJlbGFjacOzbiBlbiBBc3ltbWV0cmljUmVsYXRpb25TY29yZXIKCiAgICAj'
        'IEFycXVpdGVjdHVyYSBpbnRlcm5hIGRlbCBOb2RlRGV0ZWN0b3IKICAgIG5vZGVfY29uZmlkZW5j'
        'ZV9oaWRkZW5fZGltOiBpbnQgPSA2NAoKICAgIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5v'
        'bmU6CiAgICAgICAgaWYgc2VsZi5oaWRkZW5fZGltICUgc2VsZi5wb29sZXJfaGVhZHMgIT0gMDoK'
        'ICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgIGYiaGlkZGVuX2Rp'
        'bSAoe3NlbGYuaGlkZGVuX2RpbX0pIG11c3QgYmUgZGl2aXNpYmxlIGJ5ICIKICAgICAgICAgICAg'
        'ICAgIGYicG9vbGVyX2hlYWRzICh7c2VsZi5wb29sZXJfaGVhZHN9KSIKICAgICAgICAgICAgKQog'
        'ICAgICAgIGlmIG5vdCAwLjAgPCBzZWxmLm5vZGVfdGhyZXNob2xkIDwgMS4wOgogICAgICAgICAg'
        'ICByYWlzZSBWYWx1ZUVycm9yKGYibm9kZV90aHJlc2hvbGQgbXVzdCBiZSBpbiAoMCwxKSwgZ290'
        'IHtzZWxmLm5vZGVfdGhyZXNob2xkfSIpCiAgICAgICAgaWYgbm90IDAuMCA8IHNlbGYuZWRnZV90'
        'aHJlc2hvbGQgPCAxLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJlZGdlX3RocmVz'
        'aG9sZCBtdXN0IGJlIGluICgwLDEpLCBnb3Qge3NlbGYuZWRnZV90aHJlc2hvbGR9IikK'
    ),
    'crystallizer/node_detector.py': (
        'IiIiCmNyeXN0YWxsaXplci9ub2RlX2RldGVjdG9yLnB5IOKAlCBOb2RlRGV0ZWN0b3IKPT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpNTFAgcXVlIGNsYXNpZmlj'
        'YSBjYWRhIHBvc2ljacOzbiBkZSBsYSBzZWN1ZW5jaWEgY29tbyBub2RvIG8gbm8tbm9kby4KClBv'
        'ciBxdcOpIG5vIHVzYXIgYXR0ZW50aW9uIHBhcmEgZGV0ZWNjacOzbiBkZSBub2RvczoKICAgIExh'
        'IGRldGVjY2nDs24gZGUgbm9kb3MgZXMgdW4gcHJvYmxlbWEgUE9SIFBPU0lDScOTTiDigJQgwr9l'
        'cyBlc3RlIGNvbmNlcHRvCiAgICByZWxldmFudGUgcG9yIHPDrSBtaXNtbz8gTm8gZGVwZW5kZSBk'
        'ZSBzdSByZWxhY2nDs24gY29uIG90cm9zLgogICAgVW4gTUxQIHBvciBwb3NpY2nDs24gZXMgc3Vm'
        'aWNpZW50ZSB5IG3DoXMgZWZpY2llbnRlIHF1ZSBzZWxmLWF0dGVudGlvbi4KClRyZXMgc2FsaWRh'
        'cyBwb3IgcG9zaWNpw7NuOgogICAgbm9kZV9zY29yZSAg4oCUIMK/ZXMgZXN0ZSBjb25jZXB0byB1'
        'biBub2RvIGRlbCBncmFmbz8KICAgIHR5cGVfbG9naXRzIOKAlCDCv3F1w6kgdGlwbyBkZSBub2Rv'
        'IGVzPyAoRU5USVRZLCBFVkVOVCwgU1RBVEUsIC4uLikKICAgIGNvbmZpZGVuY2UgIOKAlCDCv2N1'
        'w6FudG8gY29uZsOtYSBlbCBkZXRlY3RvciBlbiBzdSBwcm9waWEgcHJlZGljY2nDs24/CgpMYSBz'
        'ZXBhcmFjacOzbiBkZSBgbm9kZV9zY29yZWAgeSBgY29uZmlkZW5jZWAgcGVybWl0ZToKICAgIC0g'
        'Tm9kbyBjb24gYWx0YSBwcm9iYWJpbGlkYWQgcGVybyBiYWphIGNvbmZpYW56YSAoaW5jZXJ0aWR1'
        'bWJyZSBnZW51aW5hKQogICAgLSBOb2RvIGNvbiBiYWphIHByb2JhYmlsaWRhZCBwZXJvIGFsdGEg'
        'Y29uZmlhbnphIChkZXRlY3RhIHF1ZSBOTyBlcyBub2RvKQoiIiIKCmZyb20gX19mdXR1cmVfXyBp'
        'bXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gdHlwaW5nIGltcG9ydCBUdXBsZQoKaW1wb3J0IHRvcmNo'
        'CmltcG9ydCB0b3JjaC5ubiBhcyBubgoKZnJvbSAuY29uZmlnIGltcG9ydCBDcnlzdGFsbGl6ZXJD'
        'b25maWcKCgpjbGFzcyBOb2RlRGV0ZWN0b3Iobm4uTW9kdWxlKToKICAgICIiIgogICAgTUxQIHF1'
        'ZSBhc2lnbmEgdW4gcHVudGFqZSBkZSBub2RvIHkgdGlwbyBhIGNhZGEgcG9zaWNpw7NuLgoKICAg'
        'IFVzbzoKICAgICAgICBjZmcgICAgID0gQ3J5c3RhbGxpemVyQ29uZmlnKCkKICAgICAgICBkZXRl'
        'Y3RvciA9IE5vZGVEZXRlY3RvcihjZmcpCiAgICAgICAgY29uY2VwdHMgPSB0b3JjaC5yYW5kbigy'
        'LCA2NCwgMjU2KSAgICAgICAjIFtCLCBMLCBEXQogICAgICAgIHNjb3JlcywgdHlwZV9sb2dpdHMs'
        'IGNvbmYgPSBkZXRlY3Rvcihjb25jZXB0cykKICAgICAgICAjIHNjb3JlczogICAgICBbMiwgNjRd'
        'ICAgICAgIOKAlCBzaWdtb2lkLCBwcm9iYWJpbGlkYWQgZGUgbm9kbwogICAgICAgICMgdHlwZV9s'
        'b2dpdHM6IFsyLCA2NCwgN10gICAg4oCUIHNpbiBhY3RpdmFjacOzbiwgcGFyYSBDcm9zc0VudHJv'
        'cHkKICAgICAgICAjIGNvbmY6ICAgICAgICBbMiwgNjRdICAgICAgIOKAlCBzaWdtb2lkLCBjb25m'
        'aWFuemEgZGVsIGRldGVjdG9yCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmln'
        'OiBDcnlzdGFsbGl6ZXJDb25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygp'
        'CiAgICAgICAgRCA9IGNvbmZpZy5oaWRkZW5fZGltCiAgICAgICAgQyA9IGNvbmZpZy5ub2RlX2Nv'
        'bmZpZGVuY2VfaGlkZGVuX2RpbQoKICAgICAgICAjIOKUgOKUgCBQdW50dWFjacOzbiBkZSBub2Rv'
        'IOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMgRG9zIGNhcGFzOiBs'
        'YSBwcmltZXJhIGV4cGFuZGUgcGFyYSBjYXB0dXJhciBpbnRlcmFjY2lvbmVzIGludGVybmFzLAog'
        'ICAgICAgICMgbGEgc2VndW5kYSBjb2xhcHNhIGEgdW4gZXNjYWxhci4gU2luIHNlc2dvIGVuIGxh'
        'IHNlZ3VuZGEgY2FwYQogICAgICAgICMgcGFyYSBldml0YXIgcXVlIGVsIGJpYXMgZW1wdWplIHRv'
        'ZG9zIGxvcyBzY29yZXMgaGFjaWEgYXJyaWJhLgogICAgICAgIHNlbGYubm9kZV9zY29yZXIgPSBu'
        'bi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoRCwgRCksCiAgICAgICAgICAgIG5u'
        'LkdFTFUoKSwKICAgICAgICAgICAgbm4uTGluZWFyKEQsIDEsIGJpYXM9RmFsc2UpLAogICAgICAg'
        'ICkKCiAgICAgICAgIyDilIDilIAgQ2xhc2lmaWNhZG9yIGRlIHRpcG8g4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSACiAgICAgICAgIyBQcm95ZWNjacOzbiBkaXJlY3RhIEQg4oaSIG5fdHlwZXM7'
        'IG5vIHNpZ21vaWQvc29mdG1heCBhcXXDrQogICAgICAgICMgKGVsIGNvbnN1bWlkb3IgYXBsaWNh'
        'IHNvZnRtYXggbyBhcmdtYXggc2Vnw7puIGVsIHVzbykKICAgICAgICBzZWxmLnR5cGVfY2xhc3Np'
        'ZmllciA9IG5uLkxpbmVhcihELCBjb25maWcubl9ub2RlX3R5cGVzKQoKICAgICAgICAjIOKUgOKU'
        'gCBFc3RpbWFkb3IgZGUgY29uZmlhbnphIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMg'
        'U2VwYXJhZG8gZGVsIHNjb3JlciBwYXJhIGRlc2Fjb3BsYXIgIsK/ZXMgbm9kbz8iIGRlICLCv2N1'
        'w6FudG8gY29uZsOtbz8iCiAgICAgICAgc2VsZi5jb25maWRlbmNlX2hlYWQgPSBubi5TZXF1ZW50'
        'aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoRCwgQyksCiAgICAgICAgICAgIG5uLkdFTFUoKSwK'
        'ICAgICAgICAgICAgbm4uTGluZWFyKEMsIDEsIGJpYXM9RmFsc2UpLAogICAgICAgICkKCiAgICAg'
        'ICAgc2VsZi5faW5pdF93ZWlnaHRzKCkKCiAgICBkZWYgX2luaXRfd2VpZ2h0cyhzZWxmKSAtPiBO'
        'b25lOgogICAgICAgIGZvciBtIGluIHNlbGYubW9kdWxlcygpOgogICAgICAgICAgICBpZiBpc2lu'
        'c3RhbmNlKG0sIG5uLkxpbmVhcik6CiAgICAgICAgICAgICAgICBubi5pbml0Lm5vcm1hbF8obS53'
        'ZWlnaHQsIHN0ZD0wLjAyKQogICAgICAgICAgICAgICAgaWYgbS5iaWFzIGlzIG5vdCBOb25lOgog'
        'ICAgICAgICAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKG0uYmlhcykKCiAgICBkZWYgZm9yd2Fy'
        'ZCgKICAgICAgICBzZWxmLAogICAgICAgIGNvbmNlcHRzOiB0b3JjaC5UZW5zb3IsCiAgICApIC0+'
        'IFR1cGxlW3RvcmNoLlRlbnNvciwgdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdOgogICAgICAg'
        'ICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGNvbmNlcHRzOiBbQiwgTCwgRF0g4oCUIGNv'
        'bmNlcHQgdmVjdG9ycyBkZWwgU3RyZWFtRW5jb2RlcgoKICAgICAgICBSZXR1cm5zOgogICAgICAg'
        'ICAgICBub2RlX3Njb3JlczogIFtCLCBMXSAgICAgICAgICAg4oCUIHNpZ21vaWQg4oiIICgwLCAx'
        'KQogICAgICAgICAgICB0eXBlX2xvZ2l0czogIFtCLCBMLCBuX3R5cGVzXSAg4oCUIHNpbiBhY3Rp'
        'dmFjacOzbgogICAgICAgICAgICBjb25maWRlbmNlOiAgIFtCLCBMXSAgICAgICAgICAg4oCUIHNp'
        'Z21vaWQg4oiIICgwLCAxKQogICAgICAgICIiIgogICAgICAgIG5vZGVfc2NvcmVzID0gdG9yY2gu'
        'c2lnbW9pZCgKICAgICAgICAgICAgc2VsZi5ub2RlX3Njb3Jlcihjb25jZXB0cykuc3F1ZWV6ZSgt'
        'MSkKICAgICAgICApICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBb'
        'QiwgTF0KCiAgICAgICAgdHlwZV9sb2dpdHMgPSBzZWxmLnR5cGVfY2xhc3NpZmllcihjb25jZXB0'
        'cykgICMgW0IsIEwsIG5fdHlwZXNdCgogICAgICAgIGNvbmZpZGVuY2UgPSB0b3JjaC5zaWdtb2lk'
        'KAogICAgICAgICAgICBzZWxmLmNvbmZpZGVuY2VfaGVhZChjb25jZXB0cykuc3F1ZWV6ZSgtMSkK'
        'ICAgICAgICApICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBbQiwg'
        'TF0KCiAgICAgICAgcmV0dXJuIG5vZGVfc2NvcmVzLCB0eXBlX2xvZ2l0cywgY29uZmlkZW5jZQo='
    ),
    'crystallizer/relation_scorer.py': (
        'IiIiCmNyeXN0YWxsaXplci9yZWxhdGlvbl9zY29yZXIucHkg4oCUIEFzeW1tZXRyaWNSZWxhdGlv'
        'blNjb3Jlcgo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09PQoKUHVudMO6YSByZWxhY2lvbmVzIGRpcmlnaWRhcyBlbnRyZSBwYXJlcyBkZSBu'
        'b2Rvcy4KCkVMIFBST0JMRU1BIENPTiBET1QtUFJPRFVDVCBBVFRFTlRJT046CiAgICBMYSBhdGVu'
        'Y2nDs24gZXN0w6FuZGFyIG1pZGUgU0lNSUxJVFVEOiBzY29yZShBLCBCKSA9IFEoQSnCt0soQikK'
        'ICAgIFBlcm8gUSB5IEsgc29uIGxhIG1pc21hIHByb3llY2Npw7NuIOKGkiBzY29yZShBLEIpID0g'
        'c2NvcmUoQixBKQogICAgTGEgYXRlbmNpw7NuIHNpbcOpdHJpY2EgTk8gcHVlZGUgcmVwcmVzZW50'
        'YXIgIkEgY2F1c2EgQiIg4omgICJCIGNhdXNhIEEiLgoKTEEgU09MVUNJw5NOIOKAlCBQUk9ZRUND'
        'SU9ORVMgQVNJTcOJVFJJQ0FTOgogICAgc291cmNlX3Byb2o6ICLCv3F1w6kgcm9sIGp1ZWdhIGVz'
        'dGUgbm9kbyBDT01PIEZVRU5URSBkZSB1bmEgcmVsYWNpw7NuPyIKICAgIHRhcmdldF9wcm9qOiAi'
        'wr9xdcOpIHJvbCBqdWVnYSBlc3RlIG5vZG8gQ09NTyBERVNUSU5PIGRlIHVuYSByZWxhY2nDs24/'
        'IgoKICAgIHNjb3JlKEHihpJCLCByKSA9IHNvdXJjZV9wcm9qX3IoQSkgwrcgdGFyZ2V0X3Byb2pf'
        'cihCKQogICAgc2NvcmUoQuKGkkEsIHIpID0gc291cmNlX3Byb2pfcihCKSDCtyB0YXJnZXRfcHJv'
        'al9yKEEpCgogICAgQ29tbyBzb3VyY2VfcHJvaiDiiaAgdGFyZ2V0X3Byb2o6CiAgICAgICAgc291'
        'cmNlX3Byb2ooQSkg4omgIHRhcmdldF9wcm9qKEEpICAgZW4gZ2VuZXJhbAogICAg4oaSIHNjb3Jl'
        'KEHihpJCKSDiiaAgc2NvcmUoQuKGkkEpICAgICAgICAgICAgIGVuIGdlbmVyYWwgICDinJMgQVNJ'
        'TcOJVFJJQ08KCiAgICBBbmFsb2fDrWEgZsOtc2ljYToKICAgICAgICBVbmEgZmxlY2hhICjihpIp'
        'IHRpZW5lIHB1bnRhIHkgY29sYSDigJQgbm8gZXMgbG8gbWlzbW8gaW52ZXJ0aXJsYS4KICAgICAg'
        'ICBzb3VyY2VfcHJvaiBjYXB0dXJhIGxhICJwdW50YSIgKGNhdXNhIGFjdGl2YSkuCiAgICAgICAg'
        'dGFyZ2V0X3Byb2ogY2FwdHVyYSBsYSAiY29sYSIgKGVmZWN0byByZWNlcHRpdm8pLgoKSU1QTEVN'
        'RU5UQUNJw5NOOgogICAgUG9yIGVmaWNpZW5jaWEsIHNlIHByb3llY3RhIGEgUsOXSCBkaW1lbnNp'
        'b25lcyBlbiB1biBzb2xvIExpbmVhciwKICAgIGx1ZWdvIHNlIHJlb3JnYW5pemEgY29tbyBSIGNh'
        'YmV6YXMgZGUgSCBkaW1lbnNpb25lcyBjYWRhIHVuYS4KICAgIEVsIHByb2R1Y3RvIGludGVyaW9y'
        'IHBvciBjYWJlemEgZGEgZWwgc2NvcmUgcG9yIHJlbGFjacOzbi4KCiAgICBGaW5hbG1lbnRlLCB1'
        'biBNTFAgcmVmaW5lciBhanVzdGEgbG9zIHNjb3JlcyBjb24gaW5mb3JtYWNpw7NuCiAgICBnbG9i'
        'YWwgZGVsIHZlY3RvciBkZSBzY29yZXMgY29tcGxldG8gKGludGVyYWNjaW9uZXMgZW50cmUgcmVs'
        'YWNpb25lcykuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0'
        'IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKZnJvbSAuY29uZmlnIGltcG9ydCBDcnlzdGFs'
        'bGl6ZXJDb25maWcKCgpjbGFzcyBBc3ltbWV0cmljUmVsYXRpb25TY29yZXIobm4uTW9kdWxlKToK'
        'ICAgICIiIgogICAgUHVudMO6YSBsYXMgUiByZWxhY2lvbmVzIGVudHJlIHRvZG9zIGxvcyBwYXJl'
        'cyAoaeKGkmopIGRlIG5vZG9zLgoKICAgIFVzbzoKICAgICAgICBjZmcgICAgPSBDcnlzdGFsbGl6'
        'ZXJDb25maWcoKQogICAgICAgIHNjb3JlciA9IEFzeW1tZXRyaWNSZWxhdGlvblNjb3JlcihjZmcp'
        'CgogICAgICAgICMgSW5wdXQgYmF0Y2hlZCBbQiwgbiwgRF06CiAgICAgICAgbm9kZXMgID0gdG9y'
        'Y2gucmFuZG4oMiwgOCwgMjU2KQogICAgICAgIGxvZ2l0cyA9IHNjb3Jlcihub2Rlcywgbm9kZXMp'
        'ICAgIyBbMiwgOCwgOCwgMTZdCgogICAgICAgICMgSW5wdXQgc2luIGJhdGNoIFtuLCBEXToKICAg'
        'ICAgICBub2RlcyAgPSB0b3JjaC5yYW5kbig4LCAyNTYpCiAgICAgICAgbG9naXRzID0gc2NvcmVy'
        'KG5vZGVzLCBub2RlcykgICAjIFs4LCA4LCAxNl0KCiAgICBBc2ltZXRyw61hIHZlcmlmaWNhYmxl'
        'OgogICAgICAgIGxvZ2l0c1tpLCBqLCA6XSDiiaAgbG9naXRzW2osIGksIDpdICAoZW4gZ2VuZXJh'
        'bCkKICAgICAgICBwb3JxdWUgc291cmNlX3Byb2og4omgIHRhcmdldF9wcm9qCiAgICAiIiIKCiAg'
        'ICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBDcnlzdGFsbGl6ZXJDb25maWcpIC0+IE5vbmU6'
        'CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgRCA9IGNvbmZpZy5oaWRkZW5fZGlt'
        'CiAgICAgICAgUiA9IGNvbmZpZy5uX3JlbGF0aW9uX3R5cGVzCiAgICAgICAgSCA9IGNvbmZpZy5y'
        'ZWxhdGlvbl9oaWRkZW5fZGltCgogICAgICAgICMgUHJveWVjY2lvbmVzIGFzaW3DqXRyaWNhczog'
        'bWlzbW8gaW5wdXQgRCwgcm9sZXMgZGlzdGludG9zCiAgICAgICAgIyBDYWRhIHVuYSBwcm9kdWNl'
        'IFIgc3ViLXZlY3RvcmVzIGRlIEggZGltZW5zaW9uZXMKICAgICAgICAjICh1biBzdWItZXNwYWNp'
        'byBwb3IgdGlwbyBkZSByZWxhY2nDs24pCiAgICAgICAgc2VsZi5zb3VyY2VfcHJvaiA9IG5uLkxp'
        'bmVhcihELCBSICogSCwgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLnRhcmdldF9wcm9qID0gbm4u'
        'TGluZWFyKEQsIFIgKiBILCBiaWFzPUZhbHNlKQoKICAgICAgICAjIFJlZmluYW1pZW50bzogYWp1'
        'c3RhIHNjb3JlcyBmaW5hbGVzIGNvbiBjb250ZXh0byBpbnRlci1yZWxhY2lvbmFsCiAgICAgICAg'
        'IyAoc2FiZXIgcXVlIGhheSBDQVVTRVMgcHVlZGUgbW9kdWxhciBsYSBjZXJ0ZXphIGRlIEVOQUJM'
        'RVMpCiAgICAgICAgc2VsZi5yZWZpbmVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4u'
        'TGluZWFyKFIsIFIgKiAyKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5M'
        'aW5lYXIoUiAqIDIsIFIpLAogICAgICAgICkKCiAgICAgICAgc2VsZi5fUiA9IFIKICAgICAgICBz'
        'ZWxmLl9IID0gSAoKICAgICAgICBzZWxmLl9pbml0X3dlaWdodHMoKQoKICAgIGRlZiBfaW5pdF93'
        'ZWlnaHRzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgIyBQcm95ZWNjaW9uZXM6IGluaXQgbm9ybWFs'
        'IGVzdMOhbmRhcgogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLnNvdXJjZV9wcm9qLndlaWdo'
        'dCwgc3RkPTAuMDIpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYudGFyZ2V0X3Byb2oud2Vp'
        'Z2h0LCBzdGQ9MC4wMikKICAgICAgICAjIFJlZmluZXI6IG5vcm1hbCArIGNlcm9zIGVuIGJpYXNl'
        'cwogICAgICAgIGZvciBtIGluIHNlbGYucmVmaW5lci5tb2R1bGVzKCk6CiAgICAgICAgICAgIGlm'
        'IGlzaW5zdGFuY2UobSwgbm4uTGluZWFyKToKICAgICAgICAgICAgICAgIG5uLmluaXQubm9ybWFs'
        'XyhtLndlaWdodCwgc3RkPTAuMDIpCiAgICAgICAgICAgICAgICBpZiBtLmJpYXMgaXMgbm90IE5v'
        'bmU6CiAgICAgICAgICAgICAgICAgICAgbm4uaW5pdC56ZXJvc18obS5iaWFzKQoKICAgIGRlZiBm'
        'b3J3YXJkKAogICAgICAgIHNlbGYsCiAgICAgICAgc291cmNlX25vZGVzOiB0b3JjaC5UZW5zb3Is'
        'ICAjIFtCLCBuLCBEXSDDsyBbbiwgRF0KICAgICAgICB0YXJnZXRfbm9kZXM6IHRvcmNoLlRlbnNv'
        'ciwgICMgW0IsIG0sIERdIMOzIFttLCBEXQogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAg'
        'IiIiCiAgICAgICAgQ29tcHV0YSBsb2dpdHMgZGUgcmVsYWNpw7NuIHBhcmEgdG9kb3MgbG9zIHBh'
        'cmVzIGRpcmlnaWRvcyAoaeKGkmopLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBzb3VyY2Vf'
        'bm9kZXM6IFtCLCBuLCBEXSDDsyBbbiwgRF0KICAgICAgICAgICAgdGFyZ2V0X25vZGVzOiBbQiwg'
        'bSwgRF0gw7MgW20sIERdCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIHJlbGF0aW9uX2xv'
        'Z2l0czogW0IsIG4sIG0sIFJdIMOzIFtuLCBtLCBSXQogICAgICAgICAgICAgICAgbG9naXRzWy4u'
        'LiwgaSwgaiwgcl0gPSBzY29yZSBkZSBxdWUgbm9kbyBpIHRpZW5lIHJlbGFjacOzbiByIGNvbiBu'
        'b2RvIGoKICAgICAgICAgICAgICAgIFNpbiBzaWdtb2lkL3NvZnRtYXgg4oCUIGVsIGNvbnN1bWlk'
        'b3IgZGVjaWRlIGxhIGFjdGl2YWNpw7NuLgogICAgICAgICIiIgogICAgICAgIGJhdGNoZWQgPSBz'
        'b3VyY2Vfbm9kZXMuZGltKCkgPT0gMwogICAgICAgIGlmIG5vdCBiYXRjaGVkOgogICAgICAgICAg'
        'ICBzb3VyY2Vfbm9kZXMgPSBzb3VyY2Vfbm9kZXMudW5zcXVlZXplKDApICAjIFsxLCBuLCBEXQog'
        'ICAgICAgICAgICB0YXJnZXRfbm9kZXMgPSB0YXJnZXRfbm9kZXMudW5zcXVlZXplKDApICAjIFsx'
        'LCBtLCBEXQoKICAgICAgICBCLCBuLCBEID0gc291cmNlX25vZGVzLnNoYXBlCiAgICAgICAgbSA9'
        'IHRhcmdldF9ub2Rlcy5zaGFwZVsxXQogICAgICAgIFIsIEggPSBzZWxmLl9SLCBzZWxmLl9ICgog'
        'ICAgICAgICMgW0IsIG4sIFIsIEhdIOKAlCBjYWRhIG5vZG8gcHJveWVjdGFkbyBlbiBzdSByb2wg'
        'ZGUgRlVFTlRFCiAgICAgICAgcyA9IHNlbGYuc291cmNlX3Byb2ooc291cmNlX25vZGVzKS52aWV3'
        'KEIsIG4sIFIsIEgpCgogICAgICAgICMgW0IsIG0sIFIsIEhdIOKAlCBjYWRhIG5vZG8gcHJveWVj'
        'dGFkbyBlbiBzdSByb2wgZGUgREVTVElOTwogICAgICAgIHQgPSBzZWxmLnRhcmdldF9wcm9qKHRh'
        'cmdldF9ub2RlcykudmlldyhCLCBtLCBSLCBIKQoKICAgICAgICAjIHNjb3JlKGnihpJqLCByKSA9'
        'IGRvdChzW2IsaSxyLDpdLCB0W2IsaixyLDpdKQogICAgICAgICMgW0IsIG4sIG0sIFJdID0gZWlu'
        'c3VtIHNvYnJlIGxhIGRpbWVuc2nDs24gSAogICAgICAgIHNjb3JlcyA9IHRvcmNoLmVpbnN1bSgi'
        'Ym5yaCxibXJoLT5ibm1yIiwgcywgdCkKCiAgICAgICAgIyBSZWZpbmFtaWVudG8gcG9zdC1wdW50'
        'dWFjacOzbgogICAgICAgIHJlZmluZWQgPSBzZWxmLnJlZmluZXIoc2NvcmVzKSAgIyBbQiwgbiwg'
        'bSwgUl0KCiAgICAgICAgcmV0dXJuIHJlZmluZWQgaWYgYmF0Y2hlZCBlbHNlIHJlZmluZWQuc3F1'
        'ZWV6ZSgwKSAgIyBbbiwgbSwgUl0K'
    ),
    'crystallizer/pooler.py': (
        'IiIiCmNyeXN0YWxsaXplci9wb29sZXIucHkg4oCUIENyb3NzQXR0ZW50aW9uUG9vbGVyCj09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKQWdyZWdhIGluZm9ybWFj'
        'acOzbiBkZWwgY29udGV4dG8gY29tcGxldG8gZW4gY2FkYSB2ZWN0b3IgZGUgbm9kbyBkZXRlY3Rh'
        'ZG8uCgpQUk9CTEVNQSBRVUUgUkVTVUVMVkU6CiAgICBFbCBTdHJlYW1FbmNvZGVyIHlhIGxlIGFz'
        'aWduw7MgdW4gdmVjdG9yIGEgY2FkYSBwb3NpY2nDs24uCiAgICBQZXJvIGVzZSB2ZWN0b3Igc29s'
        'byBjYXB0dXJhIGVsIGNvbnRleHRvIENBVVNBTCAodG9rZW5zIGFudGVyaW9yZXMsIGdyYWNpYXMg'
        'YWwgU1NNKS4KICAgIEVsIEdyYXBoQ3J5c3RhbGxpemVyIG5lY2VzaXRhIHVuIHZlY3RvciBxdWUg'
        'Y2FwdHVyZSBlbCBDT05URVhUTyBDT01QTEVUTyBkZWwgbm9kbwogICAg4oCUIHRhbnRvIGxvIHF1'
        'ZSBwcmVjZWRlIGNvbW8gbG8gcXVlIHNpZ3VlIOKAlCBwYXJhIGRlY2lkaXIgbGEgcmVsYWNpw7Nu'
        'IGNvbiBvdHJvcyBub2Rvcy4KCiAgICBFamVtcGxvOiAiRWwgZnVlZ28gW05PRE9dIHNlIGV4dGlu'
        'Z3Vpw7MiIHZzICJFbCBmdWVnbyBbTk9ET10gc2UgcHJvcGFnw7MiCiAgICBFbCBzaWduaWZpY2Fk'
        'byBkZSAiZnVlZ28iIGNvbW8gbm9kbyBkZWwgZ3JhZm8gZGVwZW5kZSBkZWwgY29udGV4dG8gYmlk'
        'aXJlY2Npb25hbC4KClNPTFVDScOTTjoKICAgIENyb3NzLWF0dGVudGlvbiBkb25kZToKICAgICAg'
        'ICBRdWVyaWVzICA9IHZlY3RvcmVzIGRlIHBvc2ljaW9uZXMgZGV0ZWN0YWRhcyBjb21vIG5vZG9z'
        'IChsbyBxdWUgcXVlcmVtb3MgZW5yaXF1ZWNlcikKICAgICAgICBLZXlzICAgICA9IHNlY3VlbmNp'
        'YSBjb21wbGV0YSBkZSBjb25jZXB0IHZlY3RvcnMgKGNvbnRleHRvIGEgY29uc3VsdGFyKQogICAg'
        'ICAgIFZhbHVlcyAgID0gc2VjdWVuY2lhIGNvbXBsZXRhIGRlIGNvbmNlcHQgdmVjdG9ycyAoaW5m'
        'b3JtYWNpw7NuIGEgYWdyZWdhcikKCiAgICBSZXN1bHRhZG86IGNhZGEgbm9kbyB0aWVuZSB1biB2'
        'ZWN0b3IgZW5yaXF1ZWNpZG8gY29uIGluZm9ybWFjacOzbiBnbG9iYWwuCiAgICBMb3MgdmVjdG9y'
        'ZXMgZGUgbm9kbyBzb24gbGEgaW5wdXQgYWwgQXN5bW1ldHJpY1JlbGF0aW9uU2NvcmVyLgoKUE9S'
        'IFFVw4kgQ1JPU1MtQVRURU5USU9OIFkgTk8gU0VMRi1BVFRFTlRJT046CiAgICBTZWxmLWF0dGVu'
        'dGlvbiBlbnRyZSBub2RvcyByZXF1ZXJpcsOtYSB5YSB0ZW5lciBsb3MgdmVjdG9yZXMgZmluYWxl'
        'cyBkZSBub2RvLgogICAgQ3Jvc3MtYXR0ZW50aW9uIGNvbiBlbCBjb250ZXh0byBjb21wbGV0byBl'
        'cyBlbCBwYXNvIHF1ZSBjb25zdHJ1eWUgZXNvcyB2ZWN0b3Jlcy4KIiIiCgpmcm9tIF9fZnV0dXJl'
        'X18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aAoKaW1wb3J0IHRvcmNoCmltcG9ydCB0'
        'b3JjaC5ubiBhcyBubgoKZnJvbSAuY29uZmlnIGltcG9ydCBDcnlzdGFsbGl6ZXJDb25maWcKCgpj'
        'bGFzcyBDcm9zc0F0dGVudGlvblBvb2xlcihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBBZ3JlZ2Eg'
        'Y29udGV4dG8gZW4gdmVjdG9yZXMgZGUgbm9kbyBtZWRpYW50ZSBjcm9zcy1hdHRlbnRpb24uCgog'
        'ICAgVXNvOgogICAgICAgIGNmZyAgICA9IENyeXN0YWxsaXplckNvbmZpZygpCiAgICAgICAgcG9v'
        'bGVyID0gQ3Jvc3NBdHRlbnRpb25Qb29sZXIoY2ZnKQogICAgICAgIHF1ZXJpZXMgPSB0b3JjaC5y'
        'YW5kbigyLCA4LCAyNTYpICAgIyBbQiwgbl9ub2RlcywgRF0KICAgICAgICBjb250ZXh0ID0gdG9y'
        'Y2gucmFuZG4oMiwgNjQsIDI1NikgICMgW0IsIEwsIERdCiAgICAgICAgb3V0ID0gcG9vbGVyKHF1'
        'ZXJpZXMsIGNvbnRleHQpICAgICAjIFtCLCBuX25vZGVzLCBEXQogICAgIiIiCgogICAgZGVmIF9f'
        'aW5pdF9fKHNlbGYsIGNvbmZpZzogQ3J5c3RhbGxpemVyQ29uZmlnKSAtPiBOb25lOgogICAgICAg'
        'IHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIEQgPSBjb25maWcuaGlkZGVuX2RpbQogICAgICAg'
        'IEggPSBjb25maWcucG9vbGVyX2hlYWRzCgogICAgICAgIHNlbGYubl9oZWFkcyA9IEgKICAgICAg'
        'ICBzZWxmLmhlYWRfZGltID0gRCAvLyBICiAgICAgICAgc2VsZi5zY2FsZSA9IG1hdGguc3FydChz'
        'ZWxmLmhlYWRfZGltKSAqKiAtMSAgIyA9IDEv4oiaaGVhZF9kaW0KCiAgICAgICAgIyBRdWVyaWVz'
        'IHZpZW5lbiBkZSBsb3Mgbm9kb3MgZGV0ZWN0YWRvcwogICAgICAgICMgS2V5cyB5IFZhbHVlcyB2'
        'aWVuZW4gZGVsIGNvbnRleHRvIGNvbXBsZXRvCiAgICAgICAgc2VsZi5xX3Byb2ogICA9IG5uLkxp'
        'bmVhcihELCBELCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYua19wcm9qICAgPSBubi5MaW5lYXIo'
        'RCwgRCwgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLnZfcHJvaiAgID0gbm4uTGluZWFyKEQsIEQs'
        'IGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5vdXRfcHJvaiAgPSBubi5MaW5lYXIoRCwgRCwgYmlh'
        'cz1GYWxzZSkKCiAgICAgICAgIyBOb3JtYWxpemFjacOzbiBwb3N0LXJlc2lkdWFsIChlc3TDoW5k'
        'YXIgZW4gYmxvcXVlcyBkZSBjcm9zcy1hdHRlbnRpb24pCiAgICAgICAgc2VsZi5ub3JtID0gbm4u'
        'TGF5ZXJOb3JtKEQpCgogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygpCgogICAgZGVmIF9pbml0'
        'X3dlaWdodHMoc2VsZikgLT4gTm9uZToKICAgICAgICAjIFByb3llY2Npb25lcyBjb24gc3RkIG1h'
        'eW9yIHF1ZSAwLjAyIHBhcmEgcXVlIGxhIGNyb3NzLWF0dGVudGlvbgogICAgICAgICMgcHJvZHV6'
        'Y2EgcmVwcmVzZW50YWNpb25lcyBkaXN0aW50YXMgcG9yIG5vZG8gZGVzZGUgZWwgaW5pY2lvLgog'
        'ICAgICAgICMgQ29uIHN0ZD0wLjAyIChkZWZhdWx0IExMTSksIFHiiYgwIOKGkiBhdGVuY2nDs24g'
        'dW5pZm9ybWUg4oaSIHRvZG9zIGxvcyBub2RvcwogICAgICAgICMgcmVjaWJlbiBlbCBtaXNtbyB2'
        'ZWN0b3IgYWdyZWdhZG8sIHBlcmRpZW5kbyBsYSBpZGVudGlkYWQgZGVsIG5vZG8uCiAgICAgICAg'
        'IyBzdGQ9MC4xIHByb2R1Y2Ugc2NvcmVzIGRlIGF0ZW5jacOzbiBjb24gdmFyaWFuemEgc3VmaWNp'
        'ZW50ZSBwYXJhIGRpZmVyZW5jaWFybG9zLgogICAgICAgIHN0ZCA9IDAuMQogICAgICAgIGZvciBw'
        'cm9qIGluIChzZWxmLnFfcHJvaiwgc2VsZi5rX3Byb2osIHNlbGYudl9wcm9qKToKICAgICAgICAg'
        'ICAgbm4uaW5pdC5ub3JtYWxfKHByb2oud2VpZ2h0LCBzdGQ9c3RkKQogICAgICAgICMgb3V0X3By'
        'b2ogbcOhcyBjb25zZXJ2YWRvcmEgcGFyYSBlc3RhYmlsaWRhZCBkZWwgcmVzaWR1YWwKICAgICAg'
        'ICBubi5pbml0Lm5vcm1hbF8oc2VsZi5vdXRfcHJvai53ZWlnaHQsIHN0ZD0wLjAyKQoKICAgIGRl'
        'ZiBmb3J3YXJkKAogICAgICAgIHNlbGYsCiAgICAgICAgbm9kZV9xdWVyaWVzOiB0b3JjaC5UZW5z'
        'b3IsICAjIFtCLCBuX25vZGVzLCBEXQogICAgICAgIGNvbnRleHQ6IHRvcmNoLlRlbnNvciwgICAg'
        'ICAgIyBbQiwgTCwgRF0KICAgICkgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIgogICAgICAg'
        'IEFyZ3M6CiAgICAgICAgICAgIG5vZGVfcXVlcmllczogW0IsIG5fbm9kZXMsIERdIOKAlCB2ZWN0'
        'b3JlcyBkZSBub2RvIChxdWVyaWVzKQogICAgICAgICAgICBjb250ZXh0OiAgICAgIFtCLCBMLCBE'
        'XSAgICAgICDigJQgc2VjdWVuY2lhIGNvbXBsZXRhIChrZXlzICsgdmFsdWVzKQoKICAgICAgICBS'
        'ZXR1cm5zOgogICAgICAgICAgICBub2RlX3ZlY3RvcnM6IFtCLCBuX25vZGVzLCBEXSDigJQgbm9k'
        'b3MgZW5yaXF1ZWNpZG9zIGNvbiBjb250ZXh0bwogICAgICAgICIiIgogICAgICAgIEIsIG4sIEQg'
        'PSBub2RlX3F1ZXJpZXMuc2hhcGUKICAgICAgICBMID0gY29udGV4dC5zaGFwZVsxXQogICAgICAg'
        'IEggPSBzZWxmLm5faGVhZHMKICAgICAgICBIZCA9IHNlbGYuaGVhZF9kaW0KCiAgICAgICAgIyBQ'
        'cm95ZWN0YXIgeSBzZXBhcmFyIGNhYmV6YXMKICAgICAgICBRID0gc2VsZi5xX3Byb2oobm9kZV9x'
        'dWVyaWVzKS52aWV3KEIsIG4sIEgsIEhkKS50cmFuc3Bvc2UoMSwgMikgICMgW0IsIEgsIG4sIEhk'
        'XQogICAgICAgIEsgPSBzZWxmLmtfcHJvaihjb250ZXh0KS52aWV3KEIsIEwsIEgsIEhkKS50cmFu'
        'c3Bvc2UoMSwgMikgICAgICAgICMgW0IsIEgsIEwsIEhkXQogICAgICAgIFYgPSBzZWxmLnZfcHJv'
        'aihjb250ZXh0KS52aWV3KEIsIEwsIEgsIEhkKS50cmFuc3Bvc2UoMSwgMikgICAgICAgICMgW0Is'
        'IEgsIEwsIEhkXQoKICAgICAgICAjIEF0ZW5jacOzbjogY2FkYSBub2RvIGNvbnN1bHRhIHRvZGEg'
        'bGEgc2VjdWVuY2lhCiAgICAgICAgYXR0bl93ZWlnaHRzID0gKFEgQCBLLnRyYW5zcG9zZSgtMiwg'
        'LTEpKSAqIHNlbGYuc2NhbGUgICMgW0IsIEgsIG4sIExdCiAgICAgICAgYXR0bl93ZWlnaHRzID0g'
        'dG9yY2guc29mdG1heChhdHRuX3dlaWdodHMsIGRpbT0tMSkgICAgICMgW0IsIEgsIG4sIExdCgog'
        'ICAgICAgICMgQWdyZWdhY2nDs24KICAgICAgICBhdHRuX291dCA9IGF0dG5fd2VpZ2h0cyBAIFYg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgW0IsIEgsIG4sIEhkXQogICAg'
        'ICAgIGF0dG5fb3V0ID0gYXR0bl9vdXQudHJhbnNwb3NlKDEsIDIpLmNvbnRpZ3VvdXMoKS52aWV3'
        'KEIsIG4sIEQpICAjIFtCLCBuLCBEXQoKICAgICAgICAjIFJlc2lkdWFsOiBjYWRhIG5vZG8gcmV0'
        'aWVuZSBzdSB2ZWN0b3IgZGUgY29uc3VsdGEgb3JpZ2luYWwuCiAgICAgICAgIyBFc3RvIGdhcmFu'
        'dGl6YSBxdWUgbm9kb3MgZGlzdGludG9zIHByb2R1emNhbiByZXByZXNlbnRhY2lvbmVzIGRpc3Rp'
        'bnRhcywKICAgICAgICAjIGluY2x1c28gY3VhbmRvIGxhIGNyb3NzLWF0dGVudGlvbiBlc3TDoSBp'
        'bmljaWFsaXphZGEgY29uIHBlc29zIHBlcXVlw7Fvcy4KICAgICAgICAjIG5vZGVfdmVjdG9yc1tp'
        'XSA9IExOKG5vZGVfcXVlcmllc1tpXSArIG91dF9wcm9qKGF0dG5fb3V0W2ldKSkKICAgICAgICBy'
        'ZXR1cm4gc2VsZi5ub3JtKG5vZGVfcXVlcmllcyArIHNlbGYub3V0X3Byb2ooYXR0bl9vdXQpKSAg'
        'ICAgICAgICMgW0IsIG4sIERdCg=='
    ),
    'crystallizer/model.py': (
        '...'
        'ICAgIH0K'
    ),
    'cre/__init__.py': (
        'IiIiCkFJT04tQyBjcmUg4oCUIENhdXNhbCBSZWFzb25pbmcgRW5naW5lLgoKTW90b3IgZGUgcmF6'
        'b25hbWllbnRvIGl0ZXJhdGl2byBiYXNhZG8gZW4gdHlwZWQgbWVzc2FnZSBwYXNzaW5nIGNvbiB3'
        'ZWlnaHQgc2hhcmluZy4KQ2F1c2FsR3JhcGggKyBub2RlIGZlYXR1cmVzIOKGkiBub2RlIGZlYXR1'
        'cmVzIHJlZmluYWRvcy4KIiIiCgpmcm9tIC5hZ2dyZWdhdG9yIGltcG9ydCBBdHRlbnRpdmVBZ2dy'
        'ZWdhdG9yCmZyb20gLmNvbmZpZyBpbXBvcnQgQ1JFQ29uZmlnCmZyb20gLmVuZ2luZSBpbXBvcnQg'
        'Q1JFT3V0cHV0LCBDYXVzYWxSZWFzb25pbmdFbmdpbmUKZnJvbSAubWVzc2FnZV9wYXNzaW5nIGlt'
        'cG9ydCBDYXVzYWxNZXNzYWdlUGFzc2luZ0xheWVyCmZyb20gLnNjcmF0Y2hfcGFkIGltcG9ydCBE'
        'aWZmZXJlbnRpYWJsZVNjcmF0Y2hQYWQsIFNjcmF0Y2hQYWRDb25maWcKCl9fYWxsX18gPSBbCiAg'
        'ICAiQXR0ZW50aXZlQWdncmVnYXRvciIsCiAgICAiQ1JFQ29uZmlnIiwKICAgICJDUkVPdXRwdXQi'
        'LAogICAgIkNhdXNhbE1lc3NhZ2VQYXNzaW5nTGF5ZXIiLAogICAgIkNhdXNhbFJlYXNvbmluZ0Vu'
        'Z2luZSIsCiAgICAiRGlmZmVyZW50aWFibGVTY3JhdGNoUGFkIiwKICAgICJTY3JhdGNoUGFkQ29u'
        'ZmlnIiwKXQo='
    ),
    'cre/config.py': (
        'IiIiCmNyZS9jb25maWcucHkg4oCUIENvbmZpZ3VyYWNpw7NuIGRlbCBDYXVzYWwgUmVhc29uaW5n'
        'IEVuZ2luZQo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09CiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBk'
        'YXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcwpjbGFzcyBDUkVDb25maWc6'
        'CiAgICAiIiIKICAgIEhpcGVycGFyw6FtZXRyb3MgZGVsIENhdXNhbFJlYXNvbmluZ0VuZ2luZS4K'
        'CiAgICBDb25maWd1cmFjacOzbiB0aW55IHBhcmEgdGVzdGluZzoKICAgICAgICBub2RlX2RpbT0y'
        'NTYsIGVkZ2VfZGltPTY0LCBtZXNzYWdlX2RpbT0xMjgsCiAgICAgICAgbl9tZXNzYWdlX2xheWVy'
        'cz0yLCBtYXhfaXRlcmF0aW9ucz0yMAoKICAgIFdFSUdIVCBTSEFSSU5HOgogICAgICAgIExvcyBu'
        'X21lc3NhZ2VfbGF5ZXJzIGNhcGFzIHNlIGNvbXBhcnRlbiBlbnRyZSBUT0RBUyBsYXMgaXRlcmFj'
        'aW9uZXMuCiAgICAgICAgVG90YWwgZGUgcGFyw6FtZXRyb3MgPSBwYXLDoW1ldHJvcyBkZSAyIGNh'
        'cGFzIChubyBkZSAyIMOXIDIwID0gNDAgY2FwYXMpLgogICAgICAgIFRvdGFsIGRlIEZMT1BzID0g'
        'MiBjYXBhcyDDlyBtYXhfaXRlcmF0aW9ucyAobcOhcyBiYXJhdG8gcXVlIDQwIGNhcGFzIGRpc3Rp'
        'bnRhcykuCgogICAgbl9yZWxhdGlvbl90eXBlczoKICAgICAgICBEZWJlIGNvaW5jaWRpciBjb24g'
        'bGVuKENBVVNBTF9SRUxBVElPTlMpID0gMTYuCiAgICAgICAgQ2FkYSByZWxhY2nDs24gdGllbmUg'
        'c3UgcHJvcGlhIGZ1bmNpw7NuIGRlIG1lbnNhamUgZW4gQ2F1c2FsTWVzc2FnZVBhc3NpbmdMYXll'
        'ci4KICAgICIiIgoKICAgICMgRGltZW5zaW9uZXMgcHJpbmNpcGFsZXMKICAgIG5vZGVfZGltOiAg'
        'ICAgIGludCA9IDI1NiAgICMgZGltZW5zacOzbiBkZSBsb3MgdmVjdG9yZXMgZGUgbm9kbwogICAg'
        'ZWRnZV9kaW06ICAgICAgaW50ID0gNjQgICAgIyBkaW1lbnNpw7NuIGRlIGxvcyB2ZWN0b3JlcyBk'
        'ZSBhcmlzdGEKICAgIG1lc3NhZ2VfZGltOiAgIGludCA9IDEyOCAgICMgZGltZW5zacOzbiBkZSBs'
        'b3MgbWVuc2FqZXMgZW50cmUgbm9kb3MKCiAgICAjIEFycXVpdGVjdHVyYQogICAgbl9tZXNzYWdl'
        'X2xheWVyczogaW50ID0gMiAgIyBjYXBhcyBkZSBNUCBwb3IgaXRlcmFjacOzbiAoY29tcGFydGlk'
        'YXMpCiAgICBtYXhfaXRlcmF0aW9uczogICBpbnQgPSAyMCAgIyBpdGVyYWNpb25lcyBkZSByZWZp'
        'bmFtaWVudG8gcG9yIGRlZmVjdG8KCiAgICAjIFZvY2FidWxhcmlvIChkZWJlIGNvaW5jaWRpciBj'
        'b24gY29yZS9ncmFwaC5weSkKICAgIG5fcmVsYXRpb25fdHlwZXM6IGludCA9IDE2ICAjIGxlbihD'
        'YXVzYWxSZWxhdGlvbikKCiAgICAjIEVzdGFiaWxpZGFkIG51bcOpcmljYQogICAgbm9ybV9lcHM6'
        'IGZsb2F0ID0gMWUtNgoKICAgIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAg'
        'ICAgaWYgc2VsZi5uX21lc3NhZ2VfbGF5ZXJzIDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVF'
        'cnJvcihmIm5fbWVzc2FnZV9sYXllcnMgbXVzdCBiZSA+PSAxLCBnb3Qge3NlbGYubl9tZXNzYWdl'
        'X2xheWVyc30iKQogICAgICAgIGlmIHNlbGYubWF4X2l0ZXJhdGlvbnMgPCAxOgogICAgICAgICAg'
        'ICByYWlzZSBWYWx1ZUVycm9yKGYibWF4X2l0ZXJhdGlvbnMgbXVzdCBiZSA+PSAxLCBnb3Qge3Nl'
        'bGYubWF4X2l0ZXJhdGlvbnN9IikKICAgICAgICBpZiBzZWxmLm1lc3NhZ2VfZGltIDwgMToKICAg'
        'ICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIm1lc3NhZ2VfZGltIG11c3QgYmUgPj0gMSwgZ290'
        'IHtzZWxmLm1lc3NhZ2VfZGltfSIpCg=='
    ),
    'cre/aggregator.py': (
        'IiIiCmNyZS9hZ2dyZWdhdG9yLnB5IOKAlCBBdHRlbnRpdmVBZ2dyZWdhdG9yCj09PT09PT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCkFncmVnYSBtZW5zYWplcyBlbnRyYW50ZXMg'
        'ZW4gY2FkYSBub2RvLCBwb25kZXJhbmRvIHBvciBpbXBvcnRhbmNpYSBhcHJlbmRpZGEuCgpQT1Ig'
        'UVXDiSBBVFRFTlRJVkUgWSBOTyBNRUFOOgogICAgTWVhbiBhZ2dyZWdhdGlvbiB0cmF0YSB0b2Rv'
        'cyBsb3MgbWVuc2FqZXMgY29tbyBpZ3VhbG1lbnRlIGltcG9ydGFudGVzLgogICAgUGVybyBlbiBy'
        'YXpvbmFtaWVudG8gY2F1c2FsOgogICAgICAgIC0gVW4gbWVuc2FqZSBkZSBDQVVTRVMgZnVlcnRl'
        'IGVzIG3DoXMgcmVsZXZhbnRlIHF1ZSB1bm8gZGUgQ09SUkVMQVRFUyBkw6liaWwKICAgICAgICAt'
        'IFVuIG1lbnNhamUgcXVlIGNvbmZpcm1hIGVsIGVzdGFkbyBhY3R1YWwgZXMgbWVub3MgdXJnZW50'
        'ZSBxdWUgdW5vIHF1ZSBsbyBjb250cmFkaWNlCiAgICAgICAgLSBFbCBub2RvIGRlc3Rpbm8gZGVi'
        'ZXLDrWEgcG9kZXIgImZpbHRyYXIiIG1lbnNhamVzIGlycmVsZXZhbnRlcwoKICAgIEVsIEF0dGVu'
        'dGl2ZUFnZ3JlZ2F0b3IgYXByZW5kZSBhIGFzaWduYXIgdW4gcGVzbyDPgyDiiIggKDAsMSkgYSBj'
        'YWRhIG1lbnNhamUKICAgIGJhc2FkbyBlbiAoY29udGVuaWRvIGRlbCBtZW5zYWplLCBlc3RhZG8g'
        'YWN0dWFsIGRlbCBub2RvIGRlc3Rpbm8pLgoKSU1QTEVNRU5UQUNJw5NOOgogICAgUGFyYSBjYWRh'
        'IG1lbnNhamUgZW50cmFudGU6CiAgICAgICAgc2NvcmUgPSBzaWdtb2lkKE1MUChbbWVuc2FqZSwg'
        'ZXN0YWRvX25vZG9fZGVzdGlub10pKQogICAgICAgIHBlc28gID0gc2NvcmUgIChubyBzb2Z0bWF4'
        'IOKAlCB2ZXIgbm90YSBhYmFqbykKCiAgICBMdWVnbzogYWdyZWdhZG8gPSBzdW0ocGVzbyDDlyBt'
        'ZW5zYWplKSAvIChzdW0ocGVzbykgKyDOtSkKCiAgICBQb3IgcXXDqSBzaWdtb2lkIHkgbm8gc29m'
        'dG1heDoKICAgICAgICBTb2Z0bWF4IG5vcm1hbGl6YSBzb2JyZSBsb3MgbWVuc2FqZXMgZGUgVU4g'
        'bm9kbyDihpIgcmVxdWllcmUgcGVyLXRhcmdldCBzZWdtZW50IHNvZnRtYXgKICAgICAgICAobm8g'
        'ZGlzcG9uaWJsZSBkaXJlY3RhbWVudGUgZW4gUHlUb3JjaCBzaW4gdG9yY2hfc2NhdHRlcikuCiAg'
        'ICAgICAgU2lnbW9pZCBkYSBwZXNvcyBpbmRlcGVuZGllbnRlcyDihpIgbGEgc3VtYSBub3JtYWxp'
        'emFkYSBwb3IgY291bnQvc3VtX3dlaWdodHMKICAgICAgICBlcyBlcXVpdmFsZW50ZSBhIHVuYSBh'
        'dGVuY2nDs24gc3VhdmUuCiAgICAgICAgQU1CQVMgYXByZW5kZW4gYSBwb25kZXJhciBtZW5zYWpl'
        'cyBwb3IgaW1wb3J0YW5jaWEg4oCUIGxhIGRpZmVyZW5jaWEgZXMgcXVlCiAgICAgICAgc2lnbW9p'
        'ZCBwdWVkZSBzaWxlbmNpYXIgdG9kb3MgbG9zIG1lbnNhamVzIChzY29yZSDihpIgMCkgc2kgc29u'
        'IGlycmVsZXZhbnRlcy4KClNDQVRURVIgUEFUVEVSTjoKICAgIExvcyBtZW5zYWplcyBlc3TDoW4g'
        'aW5kZXhhZG9zIHBvciBhcmlzdGEgKG5vIHBvciBub2RvKToKICAgICAgICBtZXNzYWdlczogICAg'
        'ICAgW0UsIG1lc3NhZ2VfZGltXQogICAgICAgIHRhcmdldF9pbmRpY2VzOiBbRV0gIOKAlCDDrW5k'
        'aWNlIGRlbCBub2RvIGRlc3Rpbm8gZGUgY2FkYSBtZW5zYWplCgogICAgU2NhdHRlciBhZGQ6IHBh'
        'cmEgY2FkYSBub2RvIGksIGFncmVnYSBtZW5zYWplcyBkb25kZSB0YXJnZXRfaW5kaWNlcyA9PSBp'
        'LgogICAgSW1wbGVtZW50YWRvIGNvbiB0b3JjaC5zY2F0dGVyX2FkZF8gKGRpc3BvbmlibGUgZW4g'
        'UHlUb3JjaCDiiaUgMS43KS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25z'
        'CgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpmcm9tIC5jb25maWcgaW1wb3J0'
        'IENSRUNvbmZpZwoKCmNsYXNzIEF0dGVudGl2ZUFnZ3JlZ2F0b3Iobm4uTW9kdWxlKToKICAgICIi'
        'IgogICAgQWdyZWdhIG1lbnNhamVzIGVudHJhbnRlcyBwb25kZXJhZG9zIHBvciBpbXBvcnRhbmNp'
        'YSBhcHJlbmRpZGEuCgogICAgVXNvOgogICAgICAgIGNmZyAgPSBDUkVDb25maWcoKQogICAgICAg'
        'IGFnZyAgPSBBdHRlbnRpdmVBZ2dyZWdhdG9yKGNmZykKICAgICAgICAjIG1lc3NhZ2VzOiBbRSwg'
        'TV0sIHRhcmdldHM6IFtFXSwgbm9kZXM6IFtOLCBEXQogICAgICAgIG91dCAgPSBhZ2cobWVzc2Fn'
        'ZXMsIHRhcmdldF9pbmRpY2VzLCBub2RlX2ZlYXR1cmVzLCBuX25vZGVzPU4pCiAgICAgICAgIyBv'
        'dXQ6IFtOLCBNXQogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbmZpZzogQ1JFQ29u'
        'ZmlnKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIE0gPSBjb25m'
        'aWcubWVzc2FnZV9kaW0KICAgICAgICBEID0gY29uZmlnLm5vZGVfZGltCgogICAgICAgICMgUHVu'
        'dMO6YSBsYSBpbXBvcnRhbmNpYSBkZSBjYWRhIG1lbnNhamUgZGFkbyBlbCBlc3RhZG8gZGVsIG5v'
        'ZG8gZGVzdGluby4KICAgICAgICAjIElucHV0OiBjb25jYXRlbmFjacOzbiBkZSBbbWVuc2FqZSwg'
        'ZXN0YWRvX2Rlc3Rpbm9dIOKGkiB1biBlc2NhbGFyIGVuICgwLDEpCiAgICAgICAgc2VsZi5hdHRu'
        'X3Njb3JlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcihNICsgRCwgTSAv'
        'LyAyKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoTSAvLyAy'
        'LCAxLCBiaWFzPUZhbHNlKSwKICAgICAgICApCgogICAgICAgICMgTm9ybWFsaXphY2nDs24gcG9z'
        'dC1hZ3JlZ2FjacOzbiBwYXJhIGVzdGFiaWxpZGFkCiAgICAgICAgc2VsZi5ub3JtID0gbm4uTGF5'
        'ZXJOb3JtKE0sIGVwcz1jb25maWcubm9ybV9lcHMpCgogICAgICAgIHNlbGYuX21lc3NhZ2VfZGlt'
        'ID0gTQogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygpCgogICAgZGVmIF9pbml0X3dlaWdodHMo'
        'c2VsZikgLT4gTm9uZToKICAgICAgICBmb3IgbSBpbiBzZWxmLm1vZHVsZXMoKToKICAgICAgICAg'
        'ICAgaWYgaXNpbnN0YW5jZShtLCBubi5MaW5lYXIpOgogICAgICAgICAgICAgICAgbm4uaW5pdC5u'
        'b3JtYWxfKG0ud2VpZ2h0LCBzdGQ9MC4wMikKICAgICAgICAgICAgICAgIGlmIG0uYmlhcyBpcyBu'
        'b3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBubi5pbml0Lnplcm9zXyhtLmJpYXMpCgogICAg'
        'ZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICBtZXNzYWdlczogICAgICAgdG9yY2gu'
        'VGVuc29yLCAgIyBbRSwgbWVzc2FnZV9kaW1dCiAgICAgICAgdGFyZ2V0X2luZGljZXM6IHRvcmNo'
        'LlRlbnNvciwgICMgW0VdICBsb25nIHRlbnNvcgogICAgICAgIG5vZGVfZmVhdHVyZXM6ICB0b3Jj'
        'aC5UZW5zb3IsICAjIFtOLCBub2RlX2RpbV0KICAgICAgICBuX25vZGVzOiAgICAgICAgaW50LAog'
        'ICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAg'
        'ICAgbWVzc2FnZXM6ICAgICAgIFtFLCBNXSDigJQgdW5vIHBvciBhcmlzdGEgZGVsIGdyYWZvCiAg'
        'ICAgICAgICAgIHRhcmdldF9pbmRpY2VzOiBbRV0gICAg4oCUIGEgcXXDqSBub2RvIHZhIGNhZGEg'
        'bWVuc2FqZQogICAgICAgICAgICBub2RlX2ZlYXR1cmVzOiAgW04sIERdIOKAlCBlc3RhZG8gYWN0'
        'dWFsIGRlIGxvcyBub2RvcyAoY29tbyBxdWVyeSkKICAgICAgICAgICAgbl9ub2RlczogICAgICAg'
        'IGludCAgICDigJQgTiAobmVjZXNhcmlvIGN1YW5kbyBFPTApCgogICAgICAgIFJldHVybnM6CiAg'
        'ICAgICAgICAgIGFnZ3JlZ2F0ZWQ6IFtOLCBNXSDigJQgbWVuc2FqZXMgYWdyZWdhZG9zIHBvciBu'
        'b2RvCiAgICAgICAgICAgICAgICAgICAgICAgIHplcm9zIHBhcmEgbm9kb3Mgc2luIG1lbnNhamVz'
        'IGVudHJhbnRlcwogICAgICAgICIiIgogICAgICAgIE0gPSBzZWxmLl9tZXNzYWdlX2RpbQogICAg'
        'ICAgIGRldmljZSA9IG5vZGVfZmVhdHVyZXMuZGV2aWNlCiAgICAgICAgZHR5cGUgID0gbm9kZV9m'
        'ZWF0dXJlcy5kdHlwZQoKICAgICAgICAjIOKUgOKUgCBDYXNvIHNpbiBhcmlzdGFzIOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIGlmIG1lc3NhZ2VzLnNoYXBlWzBd'
        'ID09IDA6CiAgICAgICAgICAgIHJldHVybiB0b3JjaC56ZXJvcyhuX25vZGVzLCBNLCBkZXZpY2U9'
        'ZGV2aWNlLCBkdHlwZT1kdHlwZSkKCiAgICAgICAgRSA9IG1lc3NhZ2VzLnNoYXBlWzBdCgogICAg'
        'ICAgICMg4pSA4pSAIDEuIEF0ZW5jacOzbjogcHVudMO6YSBjYWRhIG1lbnNhamUgdnMuIGVzdGFk'
        'byBkZWwgbm9kbyBkZXN0aW5vIOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMgdGFyZ2V0X2ZlYXR1'
        'cmVzOiBbRSwgRF0g4oCUIGVzdGFkbyBkZWwgbm9kbyBxdWUgUkVDSUJJUsOBIGNhZGEgbWVuc2Fq'
        'ZQogICAgICAgIHRhcmdldF9mZWF0cyA9IG5vZGVfZmVhdHVyZXNbdGFyZ2V0X2luZGljZXNdICAg'
        'ICAgICAgICMgW0UsIERdCiAgICAgICAgYXR0bl9pbnB1dCAgID0gdG9yY2guY2F0KFttZXNzYWdl'
        'cywgdGFyZ2V0X2ZlYXRzXSwgZGltPS0xKSAgIyBbRSwgTStEXQogICAgICAgIHJhd19zY29yZXMg'
        'ICA9IHNlbGYuYXR0bl9zY29yZXIoYXR0bl9pbnB1dCkuc3F1ZWV6ZSgtMSkgICAgICMgW0VdCiAg'
        'ICAgICAgd2VpZ2h0cyAgICAgID0gdG9yY2guc2lnbW9pZChyYXdfc2NvcmVzKSAgICAgICAgICAg'
        'ICAgICAgICAgIyBbRV0g4oiIICgwLDEpCgogICAgICAgICMg4pSA4pSAIDIuIE1lbnNhamVzIHBv'
        'bmRlcmFkb3Mg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgd2VpZ2h0ZWQgPSBtZXNzYWdlcyAq'
        'IHdlaWdodHMudW5zcXVlZXplKC0xKSAgICAgICAgICAgIyBbRSwgTV0KCiAgICAgICAgIyDilIDi'
        'lIAgMy4gU2NhdHRlci1hZGQ6IGFjdW11bGFyIG1lbnNhamVzIHBvciBub2RvIGRlc3Rpbm8g4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgIyBhZ2dy'
        'ZWdhdGVkW2ldID0gzqMgd2VpZ2h0ZWRbZV0gcGFyYSB0b2RvIGUgY29uIHRhcmdldF9pbmRpY2Vz'
        'W2VdID09IGkKICAgICAgICAjCiAgICAgICAgIyBzY2F0dGVyX2FkZF8gZXMgZGlmZXJlbmNpYWJs'
        'ZSByZXNwZWN0byBhIGB3ZWlnaHRlZGA6CiAgICAgICAgIyBkbC9kKHdlaWdodGVkW2VdKSA9IGRs'
        'L2QoYWdncmVnYXRlZFt0YXJnZXRfaW5kaWNlc1tlXV0pCiAgICAgICAgaWR4X2V4cGFuZGVkID0g'
        'dGFyZ2V0X2luZGljZXMudW5zcXVlZXplKC0xKS5leHBhbmQoRSwgTSkgICMgW0UsIE1dCiAgICAg'
        'ICAgYWdncmVnYXRlZCAgID0gdG9yY2guemVyb3Mobl9ub2RlcywgTSwgZGV2aWNlPWRldmljZSwg'
        'ZHR5cGU9ZHR5cGUpCiAgICAgICAgYWdncmVnYXRlZC5zY2F0dGVyX2FkZF8oMCwgaWR4X2V4cGFu'
        'ZGVkLCB3ZWlnaHRlZCkKCiAgICAgICAgIyDilIDilIAgNC4gTm9ybWFsaXphciBwb3Igc3VtYSBk'
        'ZSBwZXNvcyAoZW4gbHVnYXIgZGUgY291bnQpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgAogICAgICAgICMgc3VtX3dlaWdodHNbaV0gPSDOoyB3ZWlnaHRzW2VdIHBh'
        'cmEgdG9kbyBlIGNvbiB0YXJnZXRfaW5kaWNlc1tlXSA9PSBpCiAgICAgICAgc3VtX3dlaWdodHMg'
        'PSB0b3JjaC56ZXJvcyhuX25vZGVzLCBkZXZpY2U9ZGV2aWNlLCBkdHlwZT1kdHlwZSkKICAgICAg'
        'ICBzdW1fd2VpZ2h0cy5zY2F0dGVyX2FkZF8oMCwgdGFyZ2V0X2luZGljZXMsIHdlaWdodHMpCiAg'
        'ICAgICAgYWdncmVnYXRlZCAgID0gYWdncmVnYXRlZCAvIChzdW1fd2VpZ2h0cy51bnNxdWVlemUo'
        'LTEpICsgMWUtOCkKCiAgICAgICAgIyDilIDilIAgNS4gTGF5ZXJOb3JtIHBhcmEgZXN0YWJpbGlk'
        'YWQgbnVtw6lyaWNhIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHJldHVybiBzZWxm'
        'Lm5vcm0oYWdncmVnYXRlZCkgICAgICAgICAgICAgICAgICAgICAgICAgICMgW04sIE1dCg=='
    ),
    'cre/message_passing.py': (
        '...'
    ),
    'cre/scratch_pad.py': (
        '...'
    ),
    'cre/engine.py': (
        'IiIiCmNyZS9lbmdpbmUucHkg4oCUIENhdXNhbFJlYXNvbmluZ0VuZ2luZQo9PT09PT09PT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT0KCkVsIG1vdG9yIGRlIHJhem9uYW1pZW50byBpdGVy'
        'YXRpdm8gZGUgQUlPTi1DLgoKUkFaT05BTUlFTlRPIENPTU8gSVRFUkFDScOTTiBDT05WRVJHRU5U'
        'RToKICAgIEVsIENSRSB0b21hIHVuIENhdXNhbEdyYXBoIGNvbiBub2RlIGZlYXR1cmVzIGluaWNp'
        'YWxlcyB5IGxvICJyZWZpbmEiCiAgICBhcGxpY2FuZG8gbWVzc2FnZSBwYXNzaW5nIGl0ZXJhdGl2'
        'YW1lbnRlIGNvbiBXRUlHSFQgU0hBUklORy4KCiAgICBObyBlcyB1biB0cmFuc2Zvcm1lciBxdWUg'
        'cGFzYSBOIHZlY2VzIHBvciBOIGNhcGFzIERJU1RJTlRBUy4KICAgIEVzIGVsIG1pc21vIHNpc3Rl'
        'bWEgZGluw6FtaWNvIChtaXNtb3MgcGVzb3MpIGFwbGljYWRvIE4gdmVjZXMg4oCUCiAgICBjb21v'
        'IHJlc29sdmVyIHVuYSBFRE8gbnVtw6lyaWNhbWVudGUgbyBjb21vIGVsIGFsZ29yaXRtbyBkZQog'
        'ICAgYmVsaWVmIHByb3BhZ2F0aW9uIGVuIGdyYWZvcyBwcm9iYWJpbMOtc3RpY29zLgoKICAgIEFu'
        'YWxvZ8OtYToKICAgICAgICBUcmFuc2Zvcm1lcjogbGVlciBlbCBwcm9ibGVtYSwgcmVzcG9uZGVy'
        'IGRlIHVuYSB2ZXoKICAgICAgICBDUkU6ICAgICAgICAgbGVlciBlbCBwcm9ibGVtYSwgaXRlcmFy'
        'IGhhc3RhIHF1ZSBjb252ZXJnZQoKV0VJR0hUIFNIQVJJTkc6CiAgICBMb3MgbWlzbW9zIG5fbWVz'
        'c2FnZV9sYXllcnMgc2UgcmV1c2FuIGVuIGNhZGEgaXRlcmFjacOzbjoKICAgICAgICBpdGVyYWNp'
        'w7NuIDE6IGxheWVyc1swXSwgbGF5ZXJzWzFdCiAgICAgICAgaXRlcmFjacOzbiAyOiBsYXllcnNb'
        'MF0sIGxheWVyc1sxXSAg4oaQIG1pc21vcyBwZXNvcwogICAgICAgIC4uLgogICAgICAgIGl0ZXJh'
        'Y2nDs24gMjA6IGxheWVyc1swXSwgbGF5ZXJzWzFdICDihpAgbWlzbW9zIHBlc29zCgogICAgSW1w'
        'bGljYWNpw7NuOiAyMCBpdGVyYWNpb25lcyBubyBjdWVzdGFuIDIweCBtw6FzIHBhcsOhbWV0cm9z'
        'LgogICAgQ3Vlc3RhbiAyMHggbcOhcyBGTE9QcywgcXVlIGVzIG3DoXMgYmFyYXRvIChtZW1vcmlh'
        'IGNvbnN0YW50ZSwgY29tcHV0ZSBsaW5lYWwpLgoKICAgIEltcGxpY2FjacOzbiBwYXJhIGVudHJl'
        'bmFtaWVudG86IGJhY2twcm9wIGEgdHJhdsOpcyBkZSBtdWNoYXMgaXRlcmFjaW9uZXMKICAgIHB1'
        'ZWRlIGNhdXNhciBncmFkaWVudHMgdmFuaXNoaW5nL2V4cGxvZGluZy4gTWl0aWdhY2nDs246CiAg'
        'ICAgICAgLSBHUlVDZWxsIGNvbW8gdXBkYXRlciAoZ2F0ZXMgbmF0dXJhbG1lbnRlIGFtb3J0aWd1'
        'YW4gZ3JhZGllbnRlcykKICAgICAgICAtIExheWVyTm9ybSBkZXNwdcOpcyBkZSBjYWRhIHVwZGF0'
        'ZSAobm9ybWFsaXphIGFjdGl2YWNpb25lcykKICAgICAgICAtIFJlc2lkdWFsIGVuIGVkZ2UgdXBk'
        'YXRlcgoKSU5QVVQgRk9STUFUOgogICAgLSBncmFwaDogQ2F1c2FsR3JhcGgg4oCUIGVzdHJ1Y3R1'
        'cmEgZGlzY3JldGEgKG5vZG9zLCBhcmlzdGFzLCByZWxhY2lvbmVzKQogICAgLSBub2RlX2ZlYXR1'
        'cmVzOiBUZW5zb3IgW04sIG5vZGVfZGltXSDigJQgcmVwcmVzZW50YWNpb25lcyBpbmljaWFsZXMg'
        'ZGUgbm9kb3MKICAgICAgVXN1YWxtZW50ZSB2aWVuZSBkZWwgb3V0cHV0IGRlbCBHcmFwaENyeXN0'
        'YWxsaXplciAoY29uY2VwdCB2ZWN0b3JzIHByb3llY3RhZG9zKQoKT1VUUFVUOgogICAgQ1JFT3V0'
        'cHV0IGNvbjoKICAgIC0gbm9kZV9mZWF0dXJlczogW04sIG5vZGVfZGltXSDigJQgcmVwcmVzZW50'
        'YWNpb25lcyByZWZpbmFkYXMgKHBhcmEgU2VxdWVuY2VEZWNvZGVyKQogICAgLSBlZGdlX2ZlYXR1'
        'cmVzOiBbRSwgZWRnZV9kaW1dIOKAlCByZXByZXNlbnRhY2lvbmVzIHJlZmluYWRhcyBkZSBhcmlz'
        'dGFzCiAgICAtIGl0ZXJhdGlvbnNfcnVuOiBpbnQg4oCUIGl0ZXJhY2lvbmVzIGVqZWN1dGFkYXMK'
        'Ck5PVEEgU09CUkUgTE8gUVVFIEZBTFRBIChwb3IgZGlzZcOxbyk6CiAgICBFc3RhIGltcGxlbWVu'
        'dGFjacOzbiBlcyBlbCBDT1JFIGRlbCBtZXNzYWdlIHBhc3NpbmcuCiAgICBMbyBzaWd1aWVudGUg'
        'c2UgaW1wbGVtZW50YXLDoSBlbiBwYXNvcyBzZXBhcmFkb3M6CiAgICAgICAgLSBFbmVyZ3lGdW5j'
        'dGlvbiAoc2VjY2nDs24gNS4yIGRlbCBwbGFuKSDihpIgcGFyYSBndWlhciBlbCByZWZpbmFtaWVu'
        'dG8KICAgICAgICAtIENvbnZlcmdlbmNlR2F0ZSDihpIgcGFyYWRhIGRpbsOhbWljYSBjdWFuZG8g'
        'bGEgZW5lcmfDrWEgY29udmVyZ2UKICAgICAgICAtIE1ldGFSZWFzb25lciDihpIgbW9kaWZpY2Fj'
        'acOzbiB0b3BvbMOzZ2ljYSBkZWwgZ3JhZm8KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFu'
        'bm90YXRpb25zCgppbXBvcnQgc3lzCmltcG9ydCBvcwpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0'
        'aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShfX2ZpbGVfXykpKQoKZnJvbSBkYXRhY2xhc3NlcyBp'
        'bXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBUWVBFX0NIRUNLSU5HLCBMaXN0LCBP'
        'cHRpb25hbAoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKZnJvbSBjb3JlLmdy'
        'YXBoIGltcG9ydCBDQVVTQUxfUkVMQVRJT05TLCBDYXVzYWxHcmFwaApmcm9tIC5jb25maWcgaW1w'
        'b3J0IENSRUNvbmZpZwpmcm9tIC5tZXNzYWdlX3Bhc3NpbmcgaW1wb3J0IENhdXNhbE1lc3NhZ2VQ'
        'YXNzaW5nTGF5ZXIKCmlmIFRZUEVfQ0hFQ0tJTkc6CiAgICBmcm9tIC5zY3JhdGNoX3BhZCBpbXBv'
        'cnQgRGlmZmVyZW50aWFibGVTY3JhdGNoUGFkCgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBPVVRQVVQgQ09OVEFJTkVSCiMg'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSACgpAZGF0YWNsYXNzCmNsYXNzIENSRU91dHB1dDoKICAgICIiIgogICAgUmVzdWx0YWRvIGRl'
        'bCBDYXVzYWxSZWFzb25pbmdFbmdpbmUuCgogICAgbm9kZV9mZWF0dXJlczogW04sIG5vZGVfZGlt'
        'XSAgICDigJQgcmVwcmVzZW50YWNpb25lcyByZWZpbmFkYXMgKGVudHJhZGEgYWwgU0QpCiAgICBl'
        'ZGdlX2ZlYXR1cmVzOiBbRSwgZWRnZV9kaW1dICAgIOKAlCByZXByZXNlbnRhY2lvbmVzIHJlZmlu'
        'YWRhcyBkZSBhcmlzdGFzCiAgICBpdGVyYXRpb25zX3J1bjogaW50ICAgICAgICAgICAgIOKAlCBp'
        'dGVyYWNpb25lcyBlamVjdXRhZGFzCiAgICBsYXllcl9vdXRwdXRzOiAgTGlzdFtUZW5zb3JdICAg'
        'IOKAlCBbaXRlciDDlyBsYXllcl0gc25hcHNob3RzIHBhcmEgYW7DoWxpc2lzL3Rlc3RzCiAgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKHNvbG8gc2UgbGxlbmEgc2kgcmV0dXJu'
        'X2hpc3Rvcnk9VHJ1ZSkKICAgICIiIgogICAgbm9kZV9mZWF0dXJlczogIHRvcmNoLlRlbnNvcgog'
        'ICAgZWRnZV9mZWF0dXJlczogIHRvcmNoLlRlbnNvcgogICAgaXRlcmF0aW9uc19ydW46IGludAog'
        'ICAgbGF5ZXJfb3V0cHV0czogIExpc3RbdG9yY2guVGVuc29yXSAgIyB2YWPDrWEgcG9yIGRlZmVj'
        'dG8KCgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgAojIENBVVNBTCBSRUFTT05JTkcgRU5HSU5FCiMg4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpjbGFzcyBDYXVzYWxS'
        'ZWFzb25pbmdFbmdpbmUobm4uTW9kdWxlKToKICAgICIiIgogICAgTW90b3IgZGUgcmF6b25hbWll'
        'bnRvIGl0ZXJhdGl2byBjb24gd2VpZ2h0IHNoYXJpbmcuCgogICAgVXNvOgogICAgICAgIGNvbmZp'
        'ZyA9IENSRUNvbmZpZygpCiAgICAgICAgZW5naW5lID0gQ2F1c2FsUmVhc29uaW5nRW5naW5lKGNv'
        'bmZpZykKCiAgICAgICAgIyBncmFwaCBjb24gNSBub2RvcyB5IDggYXJpc3RhcwogICAgICAgIG5v'
        'ZGVfZmVhdHMgPSB0b3JjaC5yYW5kbig1LCAyNTYpICAgIyBmZWF0dXJlcyBpbmljaWFsZXMKICAg'
        'ICAgICBvdXRwdXQgPSBlbmdpbmUoZ3JhcGgsIG5vZGVfZmVhdHMpCiAgICAgICAgIyBvdXRwdXQu'
        'bm9kZV9mZWF0dXJlczogWzUsIDI1Nl0g4oCUIHJlZmluYWRvcyBkZXNwdcOpcyBkZSAyMCBpdGVy'
        'YWNpb25lcwoKICAgICAgICAjIE1lbm9zIGl0ZXJhY2lvbmVzICh0ZXN0aW5nIHLDoXBpZG8pCiAg'
        'ICAgICAgb3V0cHV0ID0gZW5naW5lKGdyYXBoLCBub2RlX2ZlYXRzLCBuX2l0ZXJhdGlvbnM9MykK'
        'CiAgICAgICAgIyBDb24gaGlzdG9yaWFsIHBhcmEgYW7DoWxpc2lzCiAgICAgICAgb3V0cHV0ID0g'
        'ZW5naW5lKGdyYXBoLCBub2RlX2ZlYXRzLCByZXR1cm5faGlzdG9yeT1UcnVlKQogICAgICAgICMg'
        'b3V0cHV0LmxheWVyX291dHB1dHM6IGxpc3RhIGRlIHRlbnNvcmVzIFtOLCBEXSBwb3IgY2FkYSAo'
        'aXRlciwgbGF5ZXIpCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBDUkVD'
        'b25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5j'
        'b25maWcgPSBjb25maWcKCiAgICAgICAgIyDilIDilIAgQ2FwYXMgY29tcGFydGlkYXMg4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgIyBXRUlHSFQgU0hBUklORzogZXN0'
        'YXMgbl9sYXllcnMgY2FwYXMgc2UgcmV1c2FuIGVuIENBREEgaXRlcmFjacOzbi4KICAgICAgICAj'
        'IFNvbiBsb3Mgw7puaWNvcyBwYXLDoW1ldHJvcyBkZWwgbW90b3IgZGUgcmF6b25hbWllbnRvLgog'
        'ICAgICAgIHNlbGYubGF5ZXJzOiBubi5Nb2R1bGVMaXN0ID0gbm4uTW9kdWxlTGlzdChbCiAgICAg'
        'ICAgICAgIENhdXNhbE1lc3NhZ2VQYXNzaW5nTGF5ZXIoY29uZmlnKQogICAgICAgICAgICBmb3Ig'
        'XyBpbiByYW5nZShjb25maWcubl9tZXNzYWdlX2xheWVycykKICAgICAgICBdKQoKICAgICAgICAj'
        'IOKUgOKUgCBFbWJlZGRpbmdzIGRlIGFyaXN0YXMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAg'
        'ICAgIyBDb252aWVydGUgZWwgdGlwbyBkZSByZWxhY2nDs24gKMOtbmRpY2UgZW50ZXJvKSBlbiB1'
        'biB2ZWN0b3IgY29udGludW8uCiAgICAgICAgIyBQZXJtaXRlIHF1ZSBlbCBtb2RlbG8gYXByZW5k'
        'YSByZXByZXNlbnRhY2lvbmVzIGRlbnNhcyBkZSByZWxhY2lvbmVzLgogICAgICAgICMgU2UgaW5p'
        'Y2lhbGl6YSBjb24gdW4gZW1iZWRkaW5nIHBvciB0aXBvIGRlIHJlbGFjacOzbi4KICAgICAgICBz'
        'ZWxmLmVkZ2VfdHlwZV9lbWJlZGRpbmc6IG5uLkVtYmVkZGluZyA9IG5uLkVtYmVkZGluZygKICAg'
        'ICAgICAgICAgY29uZmlnLm5fcmVsYXRpb25fdHlwZXMsIGNvbmZpZy5lZGdlX2RpbQogICAgICAg'
        'ICkKCiAgICAgICAgIyDilIDilIAgUHJveWVjdG9yIGRlIGVudHJhZGEgcGFyYSBlZGdlIHNjYWxh'
        'cnMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgIyBJbmNvcnBvcmEgc3RyZW5n'
        'dGggeSBjb25maWRlbmNlIGRlIGNhZGEgYXJpc3RhIGFsIHZlY3RvciBkZSBhcmlzdGEuCiAgICAg'
        'ICAgIyBJbnB1dDogW2VkZ2VfZGltICsgMl0gKGVtYmVkZGluZyArIHN0cmVuZ3RoICsgY29uZmlk'
        'ZW5jZSkKICAgICAgICAjIE91dHB1dDogW2VkZ2VfZGltXQogICAgICAgIHNlbGYuZWRnZV9mZWF0'
        'X3Byb2plY3Rvcjogbm4uTGluZWFyID0gbm4uTGluZWFyKAogICAgICAgICAgICBjb25maWcuZWRn'
        'ZV9kaW0gKyAyLCBjb25maWcuZWRnZV9kaW0sIGJpYXM9RmFsc2UKICAgICAgICApCgogICAgICAg'
        'IG5uLmluaXQubm9ybWFsXyhzZWxmLmVkZ2VfdHlwZV9lbWJlZGRpbmcud2VpZ2h0LCBzdGQ9MC4w'
        'MikKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5lZGdlX2ZlYXRfcHJvamVjdG9yLndlaWdo'
        'dCwgc3RkPTAuMDIpCgogICAgIyDilIDilIAgRm9yd2FyZCBwcmluY2lwYWwg4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIGZvcndhcmQoCiAgICAg'
        'ICAgc2VsZiwKICAgICAgICBncmFwaDogICAgICAgICAgIENhdXNhbEdyYXBoLAogICAgICAgIG5v'
        'ZGVfZmVhdHVyZXM6ICAgdG9yY2guVGVuc29yLCAgICAjIFtOLCBub2RlX2RpbV0KICAgICAgICBu'
        'X2l0ZXJhdGlvbnM6ICAgIGludCB8IE5vbmUgPSBOb25lLAogICAgICAgIHJldHVybl9oaXN0b3J5'
        'OiAgYm9vbCA9IEZhbHNlLAogICAgICAgIHNjcmF0Y2hfcGFkOiAgICAgIk9wdGlvbmFsW0RpZmZl'
        'cmVudGlhYmxlU2NyYXRjaFBhZF0iID0gTm9uZSwKICAgICkgLT4gQ1JFT3V0cHV0OgogICAgICAg'
        'ICIiIgogICAgICAgIEFwbGljYSBOIGl0ZXJhY2lvbmVzIGRlIG1lc3NhZ2UgcGFzc2luZyBjb24g'
        'd2VpZ2h0IHNoYXJpbmcuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGdyYXBoOiAgICAgICAg'
        'ICBDYXVzYWxHcmFwaCBjb24gc291cmNlX2lkeC90YXJnZXRfaWR4IHlhIGFzaWduYWRvcwogICAg'
        'ICAgICAgICBub2RlX2ZlYXR1cmVzOiAgW04sIG5vZGVfZGltXSDigJQgZmVhdHVyZXMgaW5pY2lh'
        'bGVzIGRlIGxvcyBub2RvcwogICAgICAgICAgICBuX2l0ZXJhdGlvbnM6ICAgbsO6bWVybyBkZSBp'
        'dGVyYWNpb25lcyAoZGVmYXVsdDogY29uZmlnLm1heF9pdGVyYXRpb25zKQogICAgICAgICAgICBy'
        'ZXR1cm5faGlzdG9yeTogc2kgVHJ1ZSwgZ3VhcmRhIHNuYXBzaG90IGRlIG5vZGVfZmVhdHVyZXMg'
        'ZW4gY2FkYSAoaXRlciwgbGF5ZXIpCiAgICAgICAgICAgIHNjcmF0Y2hfcGFkOiAgICBEaWZmZXJl'
        'bnRpYWJsZVNjcmF0Y2hQYWQgb3BjaW9uYWw7IHNpIHNlIHBhc2EsIHNlIGxsYW1hCiAgICAgICAg'
        'ICAgICAgICAgICAgICAgICAgICByZWFkKCkgeSB1cGRhdGUoKSBlbiBjYWRhIGl0ZXJhY2nDs24g'
        'dHJhcyBlbCBtZXNzYWdlIHBhc3NpbmcuCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIENS'
        'RU91dHB1dCBjb24gbm9kZV9mZWF0dXJlcyByZWZpbmFkb3MgeSBtZXRhZGF0b3MKICAgICAgICAi'
        'IiIKICAgICAgICBpZiBuX2l0ZXJhdGlvbnMgaXMgTm9uZToKICAgICAgICAgICAgbl9pdGVyYXRp'
        'b25zID0gc2VsZi5jb25maWcubWF4X2l0ZXJhdGlvbnMKCiAgICAgICAgaCA9IG5vZGVfZmVhdHVy'
        'ZXMgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIFtOLCBub2RlX2RpbV0KICAg'
        'ICAgICBlID0gc2VsZi5faW5pdF9lZGdlX2ZlYXR1cmVzKGdyYXBoLCBoLmRldmljZSwgaC5kdHlw'
        'ZSkgICMgW0UsIGVkZ2VfZGltXQoKICAgICAgICAjIOKUgOKUgCBJbmljaWFsaXphciBlc3RhZG8g'
        'ZGVsIHNjcmF0Y2ggcGFkIHNpIHNlIHByb3BvcmNpb27DsyDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIAKICAgICAgICBwYWRfc3RhdGU6ICJPcHRpb25hbFt0b3JjaC5U'
        'ZW5zb3JdIiA9IE5vbmUKICAgICAgICBpZiBzY3JhdGNoX3BhZCBpcyBub3QgTm9uZToKICAgICAg'
        'ICAgICAgcGFkX3N0YXRlID0gc2NyYXRjaF9wYWQuaW5pdF9zdGF0ZShkZXZpY2U9aC5kZXZpY2Us'
        'IGR0eXBlPWguZHR5cGUpCgogICAgICAgIGhpc3Rvcnk6IExpc3RbdG9yY2guVGVuc29yXSA9IFtd'
        'CgogICAgICAgICMg4pSA4pSAIExvb3AgZGUgaXRlcmFjaW9uZXMgY29uIFdFSUdIVCBTSEFSSU5H'
        'IOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMgTG9zIG1pc21vcyBgc2VsZi5s'
        'YXllcnNgIHNlIHJldXNhbiBlbiBjYWRhIGl0ZXJhY2nDs24uCiAgICAgICAgIyBFbCBlc3RhZG8g'
        'cXVlIGV2b2x1Y2lvbmEgc29uIGxvcyB0ZW5zb3JlcyBoLCBlIHkgcGFkX3N0YXRlIOKAlCBubyBs'
        'b3MgcGVzb3MuCiAgICAgICAgZm9yIGl0ZXJhdGlvbiBpbiByYW5nZShuX2l0ZXJhdGlvbnMpOgog'
        'ICAgICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXllcnM6CiAgICAgICAgICAgICAgICBoLCBl'
        'ID0gbGF5ZXIoaCwgZSwgZ3JhcGgpCiAgICAgICAgICAgICAgICBpZiByZXR1cm5faGlzdG9yeToK'
        'ICAgICAgICAgICAgICAgICAgICBoaXN0b3J5LmFwcGVuZChoLmRldGFjaCgpLmNsb25lKCkpCgog'
        'ICAgICAgICAgICAjIOKUgOKUgCBTY3JhdGNoIHBhZDogcmVhZCDihpIgYXVnbWVudCBub2Rlcywg'
        'dGhlbiB1cGRhdGUg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSACiAgICAgICAgICAgICMgT3JkZW46IHByaW1lcm8gbG9zIG5vZG9zIGxlZW4gZWwgY29udGV4'
        'dG8gYWN1bXVsYWRvLAogICAgICAgICAgICAjIGx1ZWdvIGVzY3JpYmVuIHN1IGVzdGFkbyBhY3R1'
        'YWwgYWwgcGFkIHBhcmEgZnV0dXJhcyBpdGVyYWNpb25lcy4KICAgICAgICAgICAgaWYgc2NyYXRj'
        'aF9wYWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBoID0gc2NyYXRjaF9wYWQucmVhZCho'
        'LCBwYWRfc3RhdGUpCiAgICAgICAgICAgICAgICBwYWRfc3RhdGUgPSBzY3JhdGNoX3BhZC51cGRh'
        'dGUoaCwgcGFkX3N0YXRlKQoKICAgICAgICByZXR1cm4gQ1JFT3V0cHV0KAogICAgICAgICAgICBu'
        'b2RlX2ZlYXR1cmVzICA9IGgsCiAgICAgICAgICAgIGVkZ2VfZmVhdHVyZXMgID0gZSwKICAgICAg'
        'ICAgICAgaXRlcmF0aW9uc19ydW4gPSBuX2l0ZXJhdGlvbnMsCiAgICAgICAgICAgIGxheWVyX291'
        'dHB1dHMgID0gaGlzdG9yeSwKICAgICAgICApCgogICAgIyDilIDilIAgSW5pY2lhbGl6YWNpw7Nu'
        'IGRlIGZlYXR1cmVzIGRlIGFyaXN0YXMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSACgogICAgZGVmIF9pbml0X2VkZ2VfZmVhdHVyZXMoCiAgICAgICAgc2VsZiwKICAg'
        'ICAgICBncmFwaDogIENhdXNhbEdyYXBoLAogICAgICAgIGRldmljZTogdG9yY2guZGV2aWNlLAog'
        'ICAgICAgIGR0eXBlOiAgdG9yY2guZHR5cGUsCiAgICApIC0+IHRvcmNoLlRlbnNvcjoKICAgICAg'
        'ICAiIiIKICAgICAgICBJbmljaWFsaXphIGVkZ2UgZmVhdHVyZXMgY29tYmluYW5kbzoKICAgICAg'
        'ICAgICAgMS4gRW1iZWRkaW5nIGFwcmVuZGlkbyBkZWwgdGlwbyBkZSByZWxhY2nDs24KICAgICAg'
        'ICAgICAgMi4gc3RyZW5ndGggeSBjb25maWRlbmNlIGRlIGNhZGEgYXJpc3RhIChkZSBDYXVzYWxF'
        'ZGdlKQoKICAgICAgICBSZXR1cm5zOiBbRSwgZWRnZV9kaW1dCiAgICAgICAgIiIiCiAgICAgICAg'
        'aWYgbm90IGdyYXBoLmVkZ2VzOgogICAgICAgICAgICByZXR1cm4gdG9yY2guemVyb3MoMCwgc2Vs'
        'Zi5jb25maWcuZWRnZV9kaW0sIGRldmljZT1kZXZpY2UsIGR0eXBlPWR0eXBlKQoKICAgICAgICAj'
        'IMONbmRpY2VzIGRlIHJlbGFjacOzbjogcGFyYSBjYWRhIGFyaXN0YSwgc3UgcG9zaWNpw7NuIGVu'
        'IENBVVNBTF9SRUxBVElPTlMKICAgICAgICByZWxfaW5kaWNlcyA9IHRvcmNoLnRlbnNvcigKICAg'
        'ICAgICAgICAgW0NBVVNBTF9SRUxBVElPTlMuaW5kZXgoZS5yZWxhdGlvbi52YWx1ZSkgZm9yIGUg'
        'aW4gZ3JhcGguZWRnZXNdLAogICAgICAgICAgICBkdHlwZT10b3JjaC5sb25nLAogICAgICAgICAg'
        'ICBkZXZpY2U9ZGV2aWNlLAogICAgICAgICkKICAgICAgICByZWxfZW1iID0gc2VsZi5lZGdlX3R5'
        'cGVfZW1iZWRkaW5nKHJlbF9pbmRpY2VzKSAgIyBbRSwgZWRnZV9kaW1dCgogICAgICAgICMgU2Nh'
        'bGFycyBwb3IgYXJpc3RhOiBzdHJlbmd0aCB5IGNvbmZpZGVuY2UKICAgICAgICBzdHJlbmd0aHMg'
        'ICA9IHRvcmNoLnRlbnNvcigKICAgICAgICAgICAgW2Uuc3RyZW5ndGggICAgZm9yIGUgaW4gZ3Jh'
        'cGguZWRnZXNdLCBkdHlwZT1kdHlwZSwgZGV2aWNlPWRldmljZQogICAgICAgICkudW5zcXVlZXpl'
        'KC0xKSAgICMgW0UsIDFdCiAgICAgICAgY29uZmlkZW5jZXMgPSB0b3JjaC50ZW5zb3IoCiAgICAg'
        'ICAgICAgIFtlLmNvbmZpZGVuY2UgIGZvciBlIGluIGdyYXBoLmVkZ2VzXSwgZHR5cGU9ZHR5cGUs'
        'IGRldmljZT1kZXZpY2UKICAgICAgICApLnVuc3F1ZWV6ZSgtMSkgICAjIFtFLCAxXQoKICAgICAg'
        'ICAjIFByb3llY3RhciB0b2RvIGp1bnRvIOKGkiBlZGdlX2RpbQogICAgICAgIGNvbWJpbmVkID0g'
        'dG9yY2guY2F0KFtyZWxfZW1iLCBzdHJlbmd0aHMsIGNvbmZpZGVuY2VzXSwgZGltPS0xKSAgIyBb'
        'RSwgZWRnZV9kaW0rMl0KICAgICAgICByZXR1cm4gc2VsZi5lZGdlX2ZlYXRfcHJvamVjdG9yKGNv'
        'bWJpbmVkKSAgICAgICAgICAgICAgICAgICAgICAgICAjIFtFLCBlZGdlX2RpbV0KCiAgICAjIOKU'
        'gOKUgCBVdGlsaWRhZGVzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBjb3VudF9wYXJhbWV0ZXJzKHNlbGYp'
        'IC0+IGludDoKICAgICAgICAiIiJOw7ptZXJvIHRvdGFsIGRlIHBhcsOhbWV0cm9zIGVudHJlbmFi'
        'bGVzLiIiIgogICAgICAgIHJldHVybiBzdW0ocC5udW1lbCgpIGZvciBwIGluIHNlbGYucGFyYW1l'
        'dGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCkKCiAgICBkZWYgcGFyYW1ldGVyX2JyZWFrZG93bihz'
        'ZWxmKSAtPiBkaWN0OgogICAgICAgICIiIkRlc2dsb3NlIGRlIHBhcsOhbWV0cm9zIHBvciBzdWIt'
        'bcOzZHVsby4iIiIKICAgICAgICBsYXllcnNfdG90YWwgPSBzdW0ocC5udW1lbCgpIGZvciBsYXll'
        'ciBpbiBzZWxmLmxheWVycyBmb3IgcCBpbiBsYXllci5wYXJhbWV0ZXJzKCkpCiAgICAgICAgcmV0'
        'dXJuIHsKICAgICAgICAgICAgImxheWVyc19zaGFyZWQiOiAgICAgICAgbGF5ZXJzX3RvdGFsLAog'
        'ICAgICAgICAgICAiZWRnZV90eXBlX2VtYmVkZGluZyI6ICBzZWxmLmVkZ2VfdHlwZV9lbWJlZGRp'
        'bmcud2VpZ2h0Lm51bWVsKCksCiAgICAgICAgICAgICJlZGdlX2ZlYXRfcHJvamVjdG9yIjogIHNl'
        'bGYuZWRnZV9mZWF0X3Byb2plY3Rvci53ZWlnaHQubnVtZWwoKSwKICAgICAgICAgICAgInRvdGFs'
        'IjogICAgICAgICAgICAgICAgc2VsZi5jb3VudF9wYXJhbWV0ZXJzKCksCiAgICAgICAgICAgICMg'
        'Tm90YTogbGF5ZXJzX3NoYXJlZCBzb24gbG9zIG1pc21vcyBwZXNvcyByZXV0aWxpemFkb3MgTiB2'
        'ZWNlcwogICAgICAgICAgICAjIOKAlCBubyBzZSBtdWx0aXBsaWNhbiBwb3IgbWF4X2l0ZXJhdGlv'
        'bnMgZW4gZWwgY29udGVvIGRlIHBhcsOhbWV0cm9zLgogICAgICAgICAgICAiZWZmZWN0aXZlX2F0'
        'X21heF9pdGVyIjogbGF5ZXJzX3RvdGFsICogc2VsZi5jb25maWcubWF4X2l0ZXJhdGlvbnMsCiAg'
        'ICAgICAgfQo='
    ),
    'decoder/__init__.py': (
        'IiIiCkFJT04tQyBkZWNvZGVyIOKAlCBTdHJlYW1EZWNvZGVyLgoKRGVjb2RpZmljYWRvciBhdXRv'
        'cmVncmVzaXZvIGNvbmRpY2lvbmFkbyBlbiBlbCBncmFmbyBjYXVzYWwgZGVsIENSRS4KQ2F1c2Fs'
        'R3JhcGgg4oaSIG5vZGVfZmVhdHVyZXMgKyB0b2tlbl9pZHMg4oaSIHRva2VuIGxvZ2l0cy4KIiIi'
        'Cgpmcm9tIC5jb25maWcgaW1wb3J0IFN0cmVhbURlY29kZXJDb25maWcKZnJvbSAuaHlicmlkX2xh'
        'eWVyIGltcG9ydCBIeWJyaWREZWNvZGVyTGF5ZXIKZnJvbSAubWV0YV9oZWFkIGltcG9ydCBNZXRh'
        'T3V0cHV0LCBPdXRwdXRNZXRhSGVhZApmcm9tIC5tb2RlbCBpbXBvcnQgRGVjb2Rlck91dHB1dCwg'
        'U3RyZWFtRGVjb2RlcgoKX19hbGxfXyA9IFsKICAgICJEZWNvZGVyT3V0cHV0IiwKICAgICJIeWJy'
        'aWREZWNvZGVyTGF5ZXIiLAogICAgIk1ldGFPdXRwdXQiLAogICAgIk91dHB1dE1ldGFIZWFkIiwK'
        'ICAgICJTdHJlYW1EZWNvZGVyIiwKICAgICJTdHJlYW1EZWNvZGVyQ29uZmlnIiwKXQo='
    ),
    'decoder/config.py': (
        'IiIiCmRlY29kZXIvY29uZmlnLnB5IOKAlCBTdHJlYW1EZWNvZGVyQ29uZmlnCiIiIgoKZnJvbSBf'
        'X2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKZnJvbSBkYXRhY2xhc3Nl'
        'cyBpbXBvcnQgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcwpjbGFzcyBTdHJlYW1EZWNvZGVyQ29uZmln'
        'OgogICAgIiIiCiAgICBDb25maWd1cmFjacOzbiBkZWwgU3RyZWFtRGVjb2Rlci4KCiAgICBUaW55'
        'ICh0ZXN0aW5nKToKICAgICAgICBoaWRkZW5fZGltPTI1Niwgbl9sYXllcnM9NCwgdm9jYWJfc2l6'
        'ZT0zMjAwMCwgbl9oZWFkcz00LCBtYXhfZ3JhcGhfbm9kZXM9MzIKCiAgICBub2RlX2RpbSBkZWJl'
        'IGNvaW5jaWRpciBjb24gQ1JFQ29uZmlnLm5vZGVfZGltIChyZXByZXNlbnRhY2lvbmVzIGRlIG5v'
        'ZG9zIGRlbCBncmFmbykuCiAgICBtYXhfc2VxX2xlbiBlcyBlbCBsw61taXRlIGRlIGxhIHRhYmxh'
        'IGRlIHBvc2ljaW9uZXMgYXByZW5kaWRhcy4KICAgICIiIgogICAgdm9jYWJfc2l6ZTogICAgICBp'
        'bnQgICA9IDMyXzAwMAogICAgaGlkZGVuX2RpbTogICAgICBpbnQgICA9IDI1NiAgICAgICMgRDog'
        'ZGltZW5zacOzbiBkZWwgbW9kZWxvCiAgICBuX2xheWVyczogICAgICAgIGludCAgID0gNCAgICAg'
        'ICAgIyBOw7ptZXJvIGRlIEh5YnJpZERlY29kZXJMYXllcnMKICAgIG5faGVhZHM6ICAgICAgICAg'
        'aW50ICAgPSA0ICAgICAgICAjIENhYmV6YXMgZGVsIGNyb3NzLWF0dGVudGlvbgogICAgbm9kZV9k'
        'aW06ICAgICAgICBpbnQgICA9IDI1NiAgICAgICMgRGltZW5zacOzbiBkZSBsb3Mgbm9kZSBmZWF0'
        'dXJlcyBkZWwgZ3JhZm8gKENSRU91dHB1dCkKICAgIG1heF9ncmFwaF9ub2RlczogaW50ICAgPSAz'
        'MiAgICAgICAjIE3DoXhpbW8gZGUgbm9kb3Mg4oCUIGNvaW5jaWRlIGNvbiBDcnlzdGFsbGl6ZXJD'
        'b25maWcubWF4X25vZGVzCiAgICBtYXhfc2VxX2xlbjogICAgIGludCAgID0gMjA0OCAgICAgIyBM'
        'w61taXRlIGRlIGxhIHRhYmxhIGRlIHBvc2ljaW9uZXMgYXByZW5kaWRhcwogICAgIyDilIDilIAg'
        'UGFyw6FtZXRyb3MgTWFtYmEgKGNvbXBhcnRpZG9zIGNvbiBlbCBlbmNvZGVyIFNTTSBpbnRlcm5v'
        'KSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHN0YXRlX2RpbTogICAg'
        'ICAgaW50ICAgPSAxNiAgICAgICAjIE46IGRpbWVuc2nDs24gZGVsIGVzdGFkbyBTU00KICAgIGV4'
        'cGFuZDogICAgICAgICAgaW50ICAgPSAyICAgICAgICAjIERfaW5uZXIgPSBleHBhbmQgw5cgRAog'
        'ICAgZF9jb252OiAgICAgICAgICBpbnQgICA9IDQgICAgICAgICMgQW5jaG8gZGUgbGEgY29udm9s'
        'dWNpw7NuIGNhdXNhbAogICAgZmZuX211bHQ6ICAgICAgICBpbnQgICA9IDQgICAgICAgICMgR2F0'
        'ZWRGRk4gaW5uZXIgPSBmZm5fbXVsdCDDlyBECiAgICBkcm9wb3V0OiAgICAgICAgIGZsb2F0ID0g'
        'MC4wCiAgICBiaWFzOiAgICAgICAgICAgIGJvb2wgID0gRmFsc2UKICAgIHJtc19lcHM6ICAgICAg'
        'ICAgZmxvYXQgPSAxZS01CiAgICBub3JtX2VwczogICAgICAgIGZsb2F0ID0gMWUtNQoKICAgIGRl'
        'ZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgc2VsZi5oaWRkZW5fZGlt'
        'ICUgc2VsZi5uX2hlYWRzICE9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAg'
        'ICAgICAgICAgICBmImhpZGRlbl9kaW0gKHtzZWxmLmhpZGRlbl9kaW19KSBtdXN0IGJlIGRpdmlz'
        'aWJsZSBieSBuX2hlYWRzICh7c2VsZi5uX2hlYWRzfSkiCiAgICAgICAgICAgICkKICAgICAgICBp'
        'ZiBzZWxmLnZvY2FiX3NpemUgPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidm9j'
        'YWJfc2l6ZSBtdXN0IGJlID49IDEsIGdvdCB7c2VsZi52b2NhYl9zaXplfSIpCiAgICAgICAgaWYg'
        'c2VsZi5tYXhfZ3JhcGhfbm9kZXMgPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYi'
        'bWF4X2dyYXBoX25vZGVzIG11c3QgYmUgPj0gMSwgZ290IHtzZWxmLm1heF9ncmFwaF9ub2Rlc30i'
        'KQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGRfaW5uZXIoc2VsZikgLT4gaW50OgogICAgICAgICIi'
        'IkRpbWVuc2nDs24gaW50ZXJuYSBkZWwgU1NNIChEX2lubmVyID0gZXhwYW5kIMOXIEQpLiIiIgog'
        'ICAgICAgIHJldHVybiBzZWxmLmV4cGFuZCAqIHNlbGYuaGlkZGVuX2RpbQoKICAgIEBwcm9wZXJ0'
        'eQogICAgZGVmIGR0X3Jhbmsoc2VsZikgLT4gaW50OgogICAgICAgICIiIlJhbmdvIGRlIM6UIOKA'
        'lCBpZ3VhbCBxdWUgZW4gZWwgZW5jb2Rlci4iIiIKICAgICAgICByZXR1cm4gbWF0aC5jZWlsKHNl'
        'bGYuaGlkZGVuX2RpbSAvIDE2KQo='
    ),
    'decoder/meta_head.py': (
        'IiIiCmRlY29kZXIvbWV0YV9oZWFkLnB5IOKAlCBPdXRwdXRNZXRhSGVhZAo9PT09PT09PT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PQoKUHJlZGljZSBtZXRhZGF0b3MgZGVsIG91dHB1dCBk'
        'ZWwgU3RyZWFtRGVjb2RlcjoKICAgIC0gY29uZmlkZW5jZTogICAgICAgICAgIMK/cXXDqSB0YW4g'
        'c2VndXJvIGVzdMOhIGVsIG1vZGVsbyBkZSBzdSByZXNwdWVzdGE/CiAgICAtIG5lZWRzX2NsYXJp'
        'ZmljYXRpb246ICDCv25lY2VzaXRhIHBlZGlyIG3DoXMgY29udGV4dG8gYWwgdXN1YXJpbz8KCkFS'
        'UVVJVEVDVFVSQToKICAgIENvbWJpbmE6CiAgICAgICAgMS4gUmVwcmVzZW50YWNpw7NuIG1lZGlh'
        'IGRlIGxvcyB0b2tlbnMgZ2VuZXJhZG9zICAobWVhbi1wb29sIHNvYnJlIEwpCiAgICAgICAgMi4g'
        'UmVwcmVzZW50YWNpw7NuIG1lZGlhIGRlbCBncmFmbyBjYXVzYWwgICAgICAgICAobWVhbi1wb29s'
        'IHNvYnJlIG5fbm9kZXMpCgogICAgTHVlZ28gYXBsaWNhIGRvcyBoZWFkcyBpbmRlcGVuZGllbnRl'
        'cyBjb24gc2lnbW9pZCBwYXJhIGVzY2FsYXIgYSBbMCwxXS4KCiAgICBJbnB1dDoKICAgICAgICBo'
        'aWRkZW46ICAgICBbQiwgTCwgaGlkZGVuX2RpbV0gICDigJQgaGlkZGVuIHN0YXRlcyBkZWwgZGVj'
        'b2RlcgogICAgICAgIGdyYXBoX3JlcHI6IFtCLCBuX25vZGVzLCBub2RlX2RpbV0g4oCUIGZlYXR1'
        'cmVzIGRlbCBncmFmbyAoZGVsIENSRSkKCiAgICBPdXRwdXQ6CiAgICAgICAgTWV0YU91dHB1dCBj'
        'b24gY29uZmlkZW5jZSBbQl0geSBuZWVkc19jbGFyaWZpY2F0aW9uIFtCXQoKTk9UQToKICAgIE9w'
        'ZXJhciBzb2JyZSBsYSBtZWRpYSBkZSB0b2tlbnMgKG5vIHNvbG8gZWwgw7psdGltbykgZXMgbcOh'
        'cyBlc3RhYmxlCiAgICBkdXJhbnRlIGVsIGVudHJlbmFtaWVudG8geSBjYXB0dXJhIGxhIGludGVu'
        'Y2nDs24gZ2xvYmFsIGRlbCB0ZXh0byBnZW5lcmFkby4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1w'
        'b3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9y'
        'dCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwg'
        'YXMgRgoKZnJvbSAuY29uZmlnIGltcG9ydCBTdHJlYW1EZWNvZGVyQ29uZmlnCgoKIyDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBP'
        'VVRQVVQgQ09OVEFJTkVSCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpAZGF0YWNsYXNzCmNsYXNzIE1ldGFPdXRwdXQ6CiAgICAi'
        'IiIKICAgIFJlc3VsdGFkbyBkZWwgT3V0cHV0TWV0YUhlYWQuCgogICAgY29uZmlkZW5jZTogICAg'
        'ICAgICAgIFtCXSDiiIggWzAsIDFdIOKAlCBjb25maWFuemEgZW4gZWwgb3V0cHV0IGdlbmVyYWRv'
        'CiAgICBuZWVkc19jbGFyaWZpY2F0aW9uOiAgW0JdIOKIiCBbMCwgMV0g4oCUIHByb2JhYmlsaWRh'
        'ZCBkZSBuZWNlc2l0YXIgY2xhcmlmaWNhY2nDs24KICAgICIiIgogICAgY29uZmlkZW5jZTogICAg'
        'ICAgICAgdG9yY2guVGVuc29yCiAgICBuZWVkc19jbGFyaWZpY2F0aW9uOiB0b3JjaC5UZW5zb3IK'
        'CgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgAojIE1FVEEgSEVBRAojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgT3V0cHV0TWV0YUhlYWQobm4uTW9kdWxl'
        'KToKICAgICIiIgogICAgUHJlZGljZSBjb25maWRlbmNlIHkgbmVlZHNfY2xhcmlmaWNhdGlvbiBk'
        'ZWwgb3V0cHV0IGdlbmVyYWRvLgoKICAgIFVzbzoKICAgICAgICBtZXRhX2hlYWQgPSBPdXRwdXRN'
        'ZXRhSGVhZChjb25maWcpCiAgICAgICAgbWV0YSA9IG1ldGFfaGVhZChoaWRkZW4sIGdyYXBoX3Jl'
        'cHIpCiAgICAgICAgIyBtZXRhLmNvbmZpZGVuY2U6ICAgICAgICAgIFtCXQogICAgICAgICMgbWV0'
        'YS5uZWVkc19jbGFyaWZpY2F0aW9uOiBbQl0KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxm'
        'LCBjb25maWc6IFN0cmVhbURlY29kZXJDb25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5f'
        'X2luaXRfXygpCiAgICAgICAgRCA9IGNvbmZpZy5oaWRkZW5fZGltCiAgICAgICAgRyA9IGNvbmZp'
        'Zy5ub2RlX2RpbQoKICAgICAgICAjIFByb3llY2Npw7NuIGNvbWJpbmFkYTogKHRva2VuX3Bvb2wg'
        '4oCWIGdyYXBoX3Bvb2wpIOKGkiBoaWRkZW4KICAgICAgICBzZWxmLnByb2ogPSBubi5MaW5lYXIo'
        'RCArIEcsIEQpCgogICAgICAgICMgRG9zIGhlYWRzIGluZGVwZW5kaWVudGVzIOKGkiBlc2NhbGFy'
        'ZXMgcG9yIGVqZW1wbG8KICAgICAgICBzZWxmLmNvbmZpZGVuY2VfaGVhZCAgICA9IG5uLkxpbmVh'
        'cihELCAxKQogICAgICAgIHNlbGYuY2xhcmlmaWNhdGlvbl9oZWFkID0gbm4uTGluZWFyKEQsIDEp'
        'CgogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygpCgogICAgZGVmIF9pbml0X3dlaWdodHMoc2Vs'
        'ZikgLT4gTm9uZToKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5wcm9qLndlaWdodCwgc3Rk'
        'PTAuMDIpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi5wcm9qLmJpYXMpCiAgICAgICAgbm4u'
        'aW5pdC5ub3JtYWxfKHNlbGYuY29uZmlkZW5jZV9oZWFkLndlaWdodCwgc3RkPTAuMDIpCiAgICAg'
        'ICAgbm4uaW5pdC56ZXJvc18oc2VsZi5jb25maWRlbmNlX2hlYWQuYmlhcykKICAgICAgICBubi5p'
        'bml0Lm5vcm1hbF8oc2VsZi5jbGFyaWZpY2F0aW9uX2hlYWQud2VpZ2h0LCBzdGQ9MC4wMikKICAg'
        'ICAgICBubi5pbml0Lnplcm9zXyhzZWxmLmNsYXJpZmljYXRpb25faGVhZC5iaWFzKQoKICAgIGRl'
        'ZiBmb3J3YXJkKAogICAgICAgIHNlbGYsCiAgICAgICAgaGlkZGVuOiAgICAgdG9yY2guVGVuc29y'
        'LCAgICMgW0IsIEwsIGhpZGRlbl9kaW1dCiAgICAgICAgZ3JhcGhfcmVwcjogdG9yY2guVGVuc29y'
        'LCAgICMgW0IsIG5fbm9kZXMsIG5vZGVfZGltXQogICAgKSAtPiBNZXRhT3V0cHV0OgogICAgICAg'
        'ICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGhpZGRlbjogICAgIFtCLCBMLCBoaWRkZW5f'
        'ZGltXSAgICDigJQgaGlkZGVuIHN0YXRlcyBmaW5hbGVzIGRlbCBkZWNvZGVyCiAgICAgICAgICAg'
        'IGdyYXBoX3JlcHI6IFtCLCBuX25vZGVzLCBub2RlX2RpbV0g4oCUIGZlYXR1cmVzIGRlbCBncmFm'
        'byBjYXVzYWwKCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgTWV0YU91dHB1dCBjb24gdGVu'
        'c29yZXMgW0JdIOKIiCBbMCwgMV0KICAgICAgICAiIiIKICAgICAgICAjIE1lYW4tcG9vbCBzb2Jy'
        'ZSBsYSBkaW1lbnNpw7NuIGRlIHNlY3VlbmNpYSAvIG5vZG9zCiAgICAgICAgaF9wb29sID0gaGlk'
        'ZGVuLm1lYW4oZGltPTEpICAgICAgIyBbQiwgRF0KICAgICAgICBnX3Bvb2wgPSBncmFwaF9yZXBy'
        'Lm1lYW4oZGltPTEpICAjIFtCLCBHXQoKICAgICAgICBjb21iaW5lZCA9IHRvcmNoLmNhdChbaF9w'
        'b29sLCBnX3Bvb2xdLCBkaW09LTEpICAjIFtCLCBEK0ddCiAgICAgICAgaCA9IEYuZ2VsdShzZWxm'
        'LnByb2ooY29tYmluZWQpKSAgICAgICAgICAgICAgICAgICMgW0IsIERdCgogICAgICAgIGNvbmZp'
        'ZGVuY2UgICAgICAgICAgPSB0b3JjaC5zaWdtb2lkKHNlbGYuY29uZmlkZW5jZV9oZWFkKGgpLnNx'
        'dWVlemUoLTEpKSAgICAgIyBbQl0KICAgICAgICBuZWVkc19jbGFyaWZpY2F0aW9uID0gdG9yY2gu'
        'c2lnbW9pZChzZWxmLmNsYXJpZmljYXRpb25faGVhZChoKS5zcXVlZXplKC0xKSkgICMgW0JdCgog'
        'ICAgICAgIHJldHVybiBNZXRhT3V0cHV0KAogICAgICAgICAgICBjb25maWRlbmNlPWNvbmZpZGVu'
        'Y2UsCiAgICAgICAgICAgIG5lZWRzX2NsYXJpZmljYXRpb249bmVlZHNfY2xhcmlmaWNhdGlvbiwK'
        'ICAgICAgICApCg=='
    ),
    'decoder/hybrid_layer.py': (
        'IiIiCmRlY29kZXIvaHlicmlkX2xheWVyLnB5IOKAlCBIeWJyaWREZWNvZGVyTGF5ZXIKPT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpDYXBhIGjDrWJyaWRhIGRl'
        'bCBTdHJlYW1EZWNvZGVyIHF1ZSBjb21iaW5hIHRyZXMgc3ViLW3Ds2R1bG9zIGVuIHNlY3VlbmNp'
        'YToKCiAgICAxLiBNYW1iYUxheWVyIChyZXV0aWxpemFkbyBkZSBlbmNvZGVyL21hbWJhX2xheWVy'
        'LnB5KQogICAgICAgUHJvY2VzYW1pZW50byBDQVVTQUwgZGUgbGEgc2VjdWVuY2lhIGRlIHRva2Vu'
        'cy4KICAgICAgIE8oTCkgZW4gbWVtb3JpYSB5IGNvbXB1dGUg4oCUIGhlcmVkYSB0b2RvcyBsb3Mg'
        'YmVuZWZpY2lvcyBkZWwgU1NNLgoKICAgIDIuIENyb3NzLWF0dGVudGlvbiBhbCBncmFmbyBjYXVz'
        'YWwgKG5uLk11bHRpaGVhZEF0dGVudGlvbikKICAgICAgIExvcyB0b2tlbnMgImNvbnN1bHRhbiIg'
        'ZWwgZ3JhZm8gcGFyYSBhbmNsYXIgc3Ugc2Vtw6FudGljYS4KICAgICAgIFEgPSB0b2tlbnMgIFtC'
        'LCBMLCBEXQogICAgICAgSyA9IFYgPSBub2RvcyBkZWwgZ3JhZm8gIFtCLCBuX25vZGVzLCBEXQog'
        'ICAgICAgTm8gaGF5IG3DoXNjYXJhIGNhdXNhbDogY2FkYSB0b2tlbiBwdWVkZSB2ZXIgdG9kb3Mg'
        'bG9zIG5vZG9zIGRlbCBncmFmby4KCiAgICAzLiBHYXRlZEZGTiAocmV1dGlsaXphZG8gZGUgZW5j'
        'b2Rlci9tYW1iYV9sYXllci5weSkKICAgICAgIFByb2Nlc2FtaWVudG8gYWRpY2lvbmFsIHBvciB0'
        'b2tlbiBkZXNwdcOpcyBkZSBpbnRlZ3JhciBlbCBjb250ZXh0byBkZWwgZ3JhZm8uCgpGTFVKTzoK'
        'ICAgIHggW0IsIEwsIERdCiAgICDihpIgTWFtYmFMYXllciAoU1NNICsgRkZOIGludGVybm8pICAg'
        'ICAgIOKGkiB4JyBbQiwgTCwgRF0KICAgIOKGkiBMTiDihpIgQ3Jvc3NBdHRuKGdyYXBoX3JlcHIp'
        'ICsgcmVzaWQgIOKGkiB4JycgW0IsIEwsIERdCiAgICDihpIgTE4g4oaSIEdhdGVkRkZOICsgcmVz'
        'aWQgICAgICAgICAgICAgICDihpIgeCcnJyBbQiwgTCwgRF0KClBPUiBRVcOJIE1BTUJBICsgQ1JP'
        'U1MtQVRURU5USU9OIChlbiB2ZXogZGUgc29sbyB0cmFuc2Zvcm1lcik6CiAgICAtIE1hbWJhIG1h'
        'bmVqYSBsYSBkZXBlbmRlbmNpYSB0ZW1wb3JhbCBhIGNvc3RvIE8oTCkKICAgIC0gQ3Jvc3MtYXR0'
        'ZW50aW9uIGFsIGdyYWZvIGVzIE8oTCDDlyBuX25vZGVzKSDigJQgY29uIG5fbm9kZXMg4omkIDMy'
        'LCBlc3RvCiAgICAgIG5vIGNyZWNlIGNvbiBsYSBsb25naXR1ZCBkZWwgdGV4dG8gZ2VuZXJhZG8K'
        'ICAgIC0gTGEgY29tYmluYWNpw7NuIGRhIGxvIG1lam9yIGRlIGFtYm9zIG11bmRvczoKICAgICAg'
        'Zmx1am8gdGVtcG9yYWwgZWZpY2llbnRlICsgY29uZGl0aW9uaW5nIHNlbcOhbnRpY28gZW4gZWwg'
        'Z3JhZm8KClJFVVRJTElaQUNJw5NOIERFIEVOQ09ERVI6CiAgICBNYW1iYUxheWVyIHkgR2F0ZWRG'
        'Rk4gc2UgaW1wb3J0YW4gZGUgZW5jb2Rlci5tYW1iYV9sYXllciBzaW4gbW9kaWZpY2FjacOzbi4K'
        'ICAgIEVsIEh5YnJpZERlY29kZXJMYXllciBjcmVhIHVuIFN0cmVhbUVuY29kZXJDb25maWcgY29t'
        'cGF0aWJsZSB1c2FuZG8KICAgIGxvcyBoaXBlcnBhcsOhbWV0cm9zIGRlbCBTdHJlYW1EZWNvZGVy'
        'Q29uZmlnLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBz'
        'eXMKaW1wb3J0IG9zCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5k'
        'aXJuYW1lKF9fZmlsZV9fKSkpCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpm'
        'cm9tIGVuY29kZXIubWFtYmFfbGF5ZXIgaW1wb3J0IEdhdGVkRkZOLCBNYW1iYUxheWVyLCBTdHJl'
        'YW1FbmNvZGVyQ29uZmlnCmZyb20gLmNvbmZpZyBpbXBvcnQgU3RyZWFtRGVjb2RlckNvbmZpZwoK'
        'CiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSACiMgSEVMUEVSOiBjb25zdHJ1aXIgU3RyZWFtRW5jb2RlckNvbmZpZyBkZXNkZSBTdHJl'
        'YW1EZWNvZGVyQ29uZmlnCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgX21ha2VfbWFtYmFfY29uZmlnKGNmZzogU3RyZWFt'
        'RGVjb2RlckNvbmZpZykgLT4gU3RyZWFtRW5jb2RlckNvbmZpZzoKICAgICIiIgogICAgQ29uc3Ry'
        'dXllIHVuIFN0cmVhbUVuY29kZXJDb25maWcgY29tcGF0aWJsZSBwYXJhIGluc3RhbmNpYXIgTWFt'
        'YmFMYXllci4KICAgIEVsIE1hbWJhTGF5ZXIgZGVsIGVuY29kZXIgZXMgaWTDqW50aWNvIGFsIHF1'
        'ZSBuZWNlc2l0YW1vcyBhcXXDrSDigJQKICAgIGxhIMO6bmljYSBkaWZlcmVuY2lhIGVzIGVsIGNv'
        'bnRleHRvIChkZWNvZGVyIHZzIGVuY29kZXIpLgogICAgIiIiCiAgICByZXR1cm4gU3RyZWFtRW5j'
        'b2RlckNvbmZpZygKICAgICAgICB2b2NhYl9zaXplICA9IGNmZy52b2NhYl9zaXplLAogICAgICAg'
        'IGhpZGRlbl9kaW0gID0gY2ZnLmhpZGRlbl9kaW0sCiAgICAgICAgbl9sYXllcnMgICAgPSAxLCAg'
        'ICAgICAgICAgICAjIHNvbG8gdXNhbW9zIDEgbGF5ZXIgYSBsYSB2ZXoKICAgICAgICBzdGF0ZV9k'
        'aW0gICA9IGNmZy5zdGF0ZV9kaW0sCiAgICAgICAgZXhwYW5kICAgICAgPSBjZmcuZXhwYW5kLAog'
        'ICAgICAgIGRfY29udiAgICAgID0gY2ZnLmRfY29udiwKICAgICAgICBmZm5fbXVsdCAgICA9IGNm'
        'Zy5mZm5fbXVsdCwKICAgICAgICBkcm9wb3V0ICAgICA9IGNmZy5kcm9wb3V0LAogICAgICAgIGJp'
        'YXMgICAgICAgID0gY2ZnLmJpYXMsCiAgICAgICAgcm1zX2VwcyAgICAgPSBjZmcucm1zX2VwcywK'
        'ICAgICkKCgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgAojIEhZQlJJRCBERUNPREVSIExBWUVSCiMg4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpjbGFzcyBIeWJyaWRE'
        'ZWNvZGVyTGF5ZXIobm4uTW9kdWxlKToKICAgICIiIgogICAgTWFtYmFMYXllciArIGNyb3NzLWF0'
        'dGVudGlvbiBhbCBncmFmbyArIEdhdGVkRkZOLgoKICAgIFVzbzoKICAgICAgICBsYXllciA9IEh5'
        'YnJpZERlY29kZXJMYXllcihjb25maWcpCiAgICAgICAgeF9vdXQgPSBsYXllcih4LCBncmFwaF9y'
        'ZXByKQogICAgICAgICMgeF9vdXQ6IFtCLCBMLCBoaWRkZW5fZGltXQoKICAgIEFyZ3M6CiAgICAg'
        'ICAgY29uZmlnOiBTdHJlYW1EZWNvZGVyQ29uZmlnCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18o'
        'c2VsZiwgY29uZmlnOiBTdHJlYW1EZWNvZGVyQ29uZmlnKSAtPiBOb25lOgogICAgICAgIHN1cGVy'
        'KCkuX19pbml0X18oKQogICAgICAgIEQgPSBjb25maWcuaGlkZGVuX2RpbQogICAgICAgIEcgPSBj'
        'b25maWcubm9kZV9kaW0KCiAgICAgICAgIyDilIDilIAgMS4gTWFtYmEgU1NNIGJsb2NrIChjb24g'
        'c3UgR2F0ZWRGRk4gaW50ZXJubykg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgbWFtYmFfY2ZnID0gX21ha2Vf'
        'bWFtYmFfY29uZmlnKGNvbmZpZykKICAgICAgICBzZWxmLm1hbWJhID0gTWFtYmFMYXllcihtYW1i'
        'YV9jZmcpCgogICAgICAgICMg4pSA4pSAIDIuIENyb3NzLWF0dGVudGlvbiBoYWNpYSBlbCBncmFm'
        'byDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBzZWxm'
        'LmNyb3NzX2F0dG5fbm9ybSA9IG5uLkxheWVyTm9ybShELCBlcHM9Y29uZmlnLm5vcm1fZXBzKQoK'
        'ICAgICAgICBzZWxmLmNyb3NzX2F0dG4gPSBubi5NdWx0aWhlYWRBdHRlbnRpb24oCiAgICAgICAg'
        'ICAgIGVtYmVkX2RpbSAgID0gRCwKICAgICAgICAgICAgbnVtX2hlYWRzICAgPSBjb25maWcubl9o'
        'ZWFkcywKICAgICAgICAgICAgYmF0Y2hfZmlyc3QgPSBUcnVlLAogICAgICAgICAgICBiaWFzICAg'
        'ICAgICA9IGNvbmZpZy5iaWFzLAogICAgICAgICAgICBkcm9wb3V0ICAgICA9IGNvbmZpZy5kcm9w'
        'b3V0LAogICAgICAgICkKCiAgICAgICAgIyBQcm95ZWN0YSBub2RlIGZlYXR1cmVzIGFsIGVzcGFj'
        'aW8gZGVsIGRlY29kZXIgc2kgZGlmaWVyZW4gZW4gZGltZW5zacOzbgogICAgICAgIHNlbGYuZ3Jh'
        'cGhfcHJvajogbm4uTW9kdWxlID0gKAogICAgICAgICAgICBubi5MaW5lYXIoRywgRCwgYmlhcz1G'
        'YWxzZSkgaWYgRyAhPSBEIGVsc2Ugbm4uSWRlbnRpdHkoKQogICAgICAgICkKCiAgICAgICAgIyDi'
        'lIDilIAgMy4gR2F0ZWRGRk4gYWRpY2lvbmFsIChwb3N0LWNyb3NzLWF0dGVudGlvbikg4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSACiAgICAgICAgc2VsZi5mZm5fbm9ybSA9IG5uLkxheWVyTm9ybShELCBlcHM9Y29uZmlnLm5v'
        'cm1fZXBzKQogICAgICAgIHNlbGYuZmZuICAgICAgPSBHYXRlZEZGTihELCBmZm5fbXVsdD1jb25m'
        'aWcuZmZuX211bHQsIGJpYXM9Y29uZmlnLmJpYXMpCgogICAgICAgIHNlbGYuZHJvcCA9ICgKICAg'
        'ICAgICAgICAgbm4uRHJvcG91dChjb25maWcuZHJvcG91dCkgaWYgY29uZmlnLmRyb3BvdXQgPiAw'
        'LjAgZWxzZSBubi5JZGVudGl0eSgpCiAgICAgICAgKQoKICAgICAgICBzZWxmLl9pbml0X3dlaWdo'
        'dHMoKQoKICAgIGRlZiBfaW5pdF93ZWlnaHRzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgIyBncmFw'
        'aF9wcm9qIHB1ZWRlIHNlciBJZGVudGl0eSBvIExpbmVhcgogICAgICAgIGlmIGlzaW5zdGFuY2Uo'
        'c2VsZi5ncmFwaF9wcm9qLCBubi5MaW5lYXIpOgogICAgICAgICAgICBubi5pbml0Lm5vcm1hbF8o'
        'c2VsZi5ncmFwaF9wcm9qLndlaWdodCwgc3RkPTAuMDIpCgogICAgZGVmIGZvcndhcmQoCiAgICAg'
        'ICAgc2VsZiwKICAgICAgICB4OiAgICAgICAgICAgdG9yY2guVGVuc29yLCAgICMgW0IsIEwsIERd'
        'CiAgICAgICAgZ3JhcGhfcmVwcjogIHRvcmNoLlRlbnNvciwgICAjIFtCLCBuX25vZGVzLCBHXQog'
        'ICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAg'
        'ICAgeDogICAgICAgICAgW0IsIEwsIGhpZGRlbl9kaW1dICAgICDigJQgaGlkZGVuIHN0YXRlcyBk'
        'ZSBsb3MgdG9rZW5zCiAgICAgICAgICAgIGdyYXBoX3JlcHI6IFtCLCBuX25vZGVzLCBub2RlX2Rp'
        'bV0g4oCUIGZlYXR1cmVzIGRlIGxvcyBub2RvcyBkZWwgZ3JhZm8KCiAgICAgICAgUmV0dXJuczoK'
        'ICAgICAgICAgICAgW0IsIEwsIGhpZGRlbl9kaW1dCiAgICAgICAgIiIiCiAgICAgICAgIyDilIDi'
        'lIAgMS4gTWFtYmFMYXllciAoU1NNIGNhdXNhbCArIEZGTiBpbnRlcm5vLCBwcmUtbm9ybSBpbnRl'
        'cm5hbWVudGUpIOKUgOKUgOKUgAogICAgICAgIHgsIF8gPSBzZWxmLm1hbWJhKHgpICAgIyBbQiwg'
        'TCwgRF0KCiAgICAgICAgIyDilIDilIAgMi4gQ3Jvc3MtYXR0ZW50aW9uIGFsIGdyYWZvIChwcmUt'
        'bm9ybSArIHJlc2lkdWFsKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIAKICAgICAgICByZXNpZHVhbCA9IHgKICAgICAgICBoID0gc2VsZi5jcm9zc19h'
        'dHRuX25vcm0oeCkgICAgICAgICAgICAgICAgICAgICMgW0IsIEwsIERdCiAgICAgICAga3YgPSBz'
        'ZWxmLmdyYXBoX3Byb2ooZ3JhcGhfcmVwcikgICAgICAgICAgICAgICAjIFtCLCBuX25vZGVzLCBE'
        'XQogICAgICAgIGF0dG5fb3V0LCBfID0gc2VsZi5jcm9zc19hdHRuKAogICAgICAgICAgICBxdWVy'
        'eSA9IGgsCiAgICAgICAgICAgIGtleSAgID0ga3YsCiAgICAgICAgICAgIHZhbHVlID0ga3YsCiAg'
        'ICAgICAgKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIFtC'
        'LCBMLCBEXQogICAgICAgIHggPSByZXNpZHVhbCArIHNlbGYuZHJvcChhdHRuX291dCkKCiAgICAg'
        'ICAgIyDilIDilIAgMy4gR2F0ZWRGRk4gKHByZS1ub3JtICsgcmVzaWR1YWwpIOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHJlc2lkdWFsID0geAogICAg'
        'ICAgIHggPSByZXNpZHVhbCArIHNlbGYuZHJvcChzZWxmLmZmbihzZWxmLmZmbl9ub3JtKHgpKSkK'
        'CiAgICAgICAgcmV0dXJuIHgK'
    ),
    'decoder/model.py': (
        'IiIiCmRlY29kZXIvbW9kZWwucHkg4oCUIFN0cmVhbURlY29kZXIKPT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09PT09PT09PQoKRWwgZGVjb2RpZmljYWRvciBhdXRvcmVncmVzaXZvIGRlIEFJT04t'
        'QyBjb25kaWNpb25hZG8gZW4gZWwgZ3JhZm8gY2F1c2FsLgoKQVJRVUlURUNUVVJBOgogICAgdG9r'
        'ZW5faWRzIFtCLCBMXQogICAgICAgIOKGkyB0b2tlbl9lbWJlZGRpbmcgKyBwb3NfZW1iZWRkaW5n'
        'CiAgICB4IFtCLCBMLCBEXQogICAgICAgIOKGkyBIeWJyaWREZWNvZGVyTGF5ZXIgw5cgbl9sYXll'
        'cnMKICAgICAgICAgIChNYW1iYUxheWVyICsgY3Jvc3MtYXR0biDihpIgZ3JhZm8gKyBHYXRlZEZG'
        'TikKICAgIHggW0IsIEwsIERdCiAgICAgICAg4oaTIGZpbmFsX25vcm0KICAgIOKUjOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUkAogICAg4pSCICBsbV9oZWFkIFtCLCBMLCB2b2NhYl9zaXplXSAgIOKG'
        'kiB0b2tlbiBsb2dpdHPilIIKICAgIOKUgiAgYW5jaG9yX2hlYWQgW0IsIEwsIG1heF9ub2Rlc13i'
        'hpIgYW5jaG9yIGxvZ2l0c+KUggogICAg4pSCICBtZXRhX2hlYWQg4oaSIGNvbmZpZGVuY2UsIG5l'
        'ZWRzX2NsYXJpZmljYXRpb24g4pSCCiAgICDilJTilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilJgKCkNP'
        'TkRJVElPTklORyBFTiBFTCBHUkFGTzoKICAgIGdyYXBoX3JlcHIgW0IsIG5fbm9kZXMsIG5vZGVf'
        'ZGltXSDigJQgdmllbmUgZGVsIENhdXNhbFJlYXNvbmluZ0VuZ2luZS4KICAgIEVuIGNhZGEgSHli'
        'cmlkRGVjb2RlckxheWVyLCBsb3MgdG9rZW5zIGhhY2VuIGNyb3NzLWF0dGVudGlvbiBhIGVzdG9z'
        'CiAgICB2ZWN0b3JlcyBkZSBub2RvLiBFc3RvIHBlcm1pdGUgYWwgZGVjb2RlciAiYW5jbGFyIiBj'
        'YWRhIHRva2VuIGEgdW4KICAgIG5vZG8gZXNwZWPDrWZpY28gZGVsIGdyYWZvIGNhdXNhbC4KCkFO'
        'Q0hPUiBIRUFEOgogICAgUHJlZGljZSBhIHF1w6kgbm9kbyBkZWwgZ3JhZm8gInBlcnRlbmVjZSIg'
        'Y2FkYSB0b2tlbiBnZW5lcmFkby4KICAgIER1cmFudGUgZWwgZW50cmVuYW1pZW50byBzZSB1c2Eg'
        'Y29tbyBzZcOxYWwgYXV4aWxpYXIgcGFyYSBmb3J6YXIKICAgIHF1ZSBsb3MgdG9rZW5zIHNlYW4g'
        'ZmllbGVzIGFsIHJhem9uYW1pZW50byBjYXVzYWwuCgpPVVRQVVQgTUVUQSBIRUFEOgogICAgUHJl'
        'ZGljZSBtZXRhZGF0b3MgZGVsIG91dHB1dDoKICAgICAgICBjb25maWRlbmNlOiAgICAgICAgICDC'
        'v3F1w6kgdGFuIHNlZ3VybyBlc3TDoSBlbCBtb2RlbG8/CiAgICAgICAgbmVlZHNfY2xhcmlmaWNh'
        'dGlvbjogwr9uZWNlc2l0YSBwZWRpciBtw6FzIGluZm9ybWFjacOzbj8KCldFSUdIVCBUWUlORzoK'
        'ICAgIGxtX2hlYWQud2VpZ2h0ID0gdG9rZW5fZW1iZWRkaW5nLndlaWdodCAocHLDoWN0aWNhIGVz'
        'dMOhbmRhciBlbiBMTXMpLgogICAgUmVkdWNlIHBhcsOhbWV0cm9zIHkgbWVqb3JhIGNvbnZlcmdl'
        'bmNpYS4KClBPU0lDSU9ORVM6CiAgICBUYWJsYSBkZSBlbWJlZGRpbmdzIGFwcmVuZGlkYSBkZSB0'
        'YW1hw7FvIG1heF9zZXFfbGVuLgogICAgTcOhcyBzaW1wbGUgcXVlIFJvUEUvQUxpQmksIHN1Zmlj'
        'aWVudGUgcGFyYSBlbCBzY29wZSBkZSBBSU9OLUMuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9y'
        'dCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwppbXBvcnQgb3MKc3lzLnBhdGguaW5zZXJ0KDAsIG9z'
        'LnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUoX19maWxlX18pKSkKCmZyb20gZGF0YWNsYXNz'
        'ZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgTGlzdAoKaW1wb3J0IHRvcmNo'
        'CmltcG9ydCB0b3JjaC5ubiBhcyBubgoKZnJvbSAuY29uZmlnIGltcG9ydCBTdHJlYW1EZWNvZGVy'
        'Q29uZmlnCmZyb20gLmh5YnJpZF9sYXllciBpbXBvcnQgSHlicmlkRGVjb2RlckxheWVyCmZyb20g'
        'Lm1ldGFfaGVhZCBpbXBvcnQgTWV0YU91dHB1dCwgT3V0cHV0TWV0YUhlYWQKCgojIOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIE9V'
        'VFBVVCBDT05UQUlORVIKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIAKCkBkYXRhY2xhc3MKY2xhc3MgRGVjb2Rlck91dHB1dDoKICAg'
        'ICIiIgogICAgUmVzdWx0YWRvIGRlbCBTdHJlYW1EZWNvZGVyLgoKICAgIGxvZ2l0czogICAgICAg'
        'ICAgICAgICBbQiwgTCwgdm9jYWJfc2l6ZV0gICAg4oCUIGRpc3RyaWJ1Y2nDs24gc29icmUgdm9j'
        'YWJ1bGFyaW8KICAgIGFuY2hvcl9sb2dpdHM6ICAgICAgICBbQiwgTCwgbWF4X2dyYXBoX25vZGVz'
        'XSDigJQgcXXDqSBub2RvIGRlbCBncmFmbyBwcm9kdWNlIGNhZGEgdG9rZW4KICAgIGNvbmZpZGVu'
        'Y2U6ICAgICAgICAgICBbQl0g4oiIIFswLDFdICAgICAgICAgIOKAlCBjb25maWFuemEgZGVsIG1v'
        'ZGVsbwogICAgbmVlZHNfY2xhcmlmaWNhdGlvbjogIFtCXSDiiIggWzAsMV0gICAgICAgICAg4oCU'
        'IHNvbGljaXRhciBtw6FzIGluZm8gYWwgdXN1YXJpbwogICAgIiIiCiAgICBsb2dpdHM6ICAgICAg'
        'ICAgICAgICB0b3JjaC5UZW5zb3IKICAgIGFuY2hvcl9sb2dpdHM6ICAgICAgIHRvcmNoLlRlbnNv'
        'cgogICAgY29uZmlkZW5jZTogICAgICAgICAgdG9yY2guVGVuc29yCiAgICBuZWVkc19jbGFyaWZp'
        'Y2F0aW9uOiB0b3JjaC5UZW5zb3IKCgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIFNUUkVBTSBERUNPREVSCiMg4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpjbGFz'
        'cyBTdHJlYW1EZWNvZGVyKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIERlY29kaWZpY2Fkb3IgYXV0'
        'b3JlZ3Jlc2l2byBjb25kaWNpb25hZG8gZW4gZWwgZ3JhZm8gY2F1c2FsLgoKICAgIFVzbzoKICAg'
        'ICAgICBjb25maWcgID0gU3RyZWFtRGVjb2RlckNvbmZpZygpCiAgICAgICAgZGVjb2RlciA9IFN0'
        'cmVhbURlY29kZXIoY29uZmlnKQoKICAgICAgICAjIGdyYXBoX3JlcHIgZGVsIENhdXNhbFJlYXNv'
        'bmluZ0VuZ2luZSAoQ1JFT3V0cHV0Lm5vZGVfZmVhdHVyZXMpCiAgICAgICAgdG9rZW5faWRzICA9'
        'IHRvcmNoLnJhbmRpbnQoMCwgY29uZmlnLnZvY2FiX3NpemUsICgyLCAxNikpCiAgICAgICAgZ3Jh'
        'cGhfcmVwciA9IHRvcmNoLnJhbmRuKDIsIDgsIGNvbmZpZy5ub2RlX2RpbSkKCiAgICAgICAgb3V0'
        'ID0gZGVjb2Rlcih0b2tlbl9pZHMsIGdyYXBoX3JlcHIpCiAgICAgICAgIyBvdXQubG9naXRzOiAg'
        'ICAgICAgIFsyLCAxNiwgMzIwMDBdCiAgICAgICAgIyBvdXQuYW5jaG9yX2xvZ2l0czogIFsyLCAx'
        'NiwgMzJdCiAgICAgICAgIyBvdXQuY29uZmlkZW5jZTogICAgIFsyXQogICAgIiIiCgogICAgZGVm'
        'IF9faW5pdF9fKHNlbGYsIGNvbmZpZzogU3RyZWFtRGVjb2RlckNvbmZpZykgLT4gTm9uZToKICAg'
        'ICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZwogICAg'
        'ICAgIEQgPSBjb25maWcuaGlkZGVuX2RpbQoKICAgICAgICAjIOKUgOKUgCBFbWJlZGRpbmdzIOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAg'
        'ICAgIHNlbGYudG9rZW5fZW1iZWRkaW5nID0gbm4uRW1iZWRkaW5nKGNvbmZpZy52b2NhYl9zaXpl'
        'LCBEKQogICAgICAgIHNlbGYucG9zX2VtYmVkZGluZyAgID0gbm4uRW1iZWRkaW5nKGNvbmZpZy5t'
        'YXhfc2VxX2xlbiwgRCkKCiAgICAgICAgIyDilIDilIAgQ2FwYXMgaMOtYnJpZGFzIOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHNlbGYubGF5ZXJzOiBu'
        'bi5Nb2R1bGVMaXN0ID0gbm4uTW9kdWxlTGlzdChbCiAgICAgICAgICAgIEh5YnJpZERlY29kZXJM'
        'YXllcihjb25maWcpCiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKGNvbmZpZy5uX2xheWVycykK'
        'ICAgICAgICBdKQoKICAgICAgICAjIOKUgOKUgCBOb3JtYWxpemFjacOzbiBmaW5hbCDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIAKICAgICAgICBzZWxmLmZpbmFsX25vcm0gPSBubi5MYXllck5v'
        'cm0oRCwgZXBzPWNvbmZpZy5ub3JtX2VwcykKCiAgICAgICAgIyDilIDilIAgSGVhZHMgZGUgc2Fs'
        'aWRhIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMgTE0g'
        'aGVhZDogcHJlZGljZSBlbCBzaWd1aWVudGUgdG9rZW4KICAgICAgICBzZWxmLmxtX2hlYWQgPSBu'
        'bi5MaW5lYXIoRCwgY29uZmlnLnZvY2FiX3NpemUsIGJpYXM9RmFsc2UpCgogICAgICAgICMgQW5j'
        'aG9yIGhlYWQ6IHF1w6kgbm9kbyBkZWwgZ3JhZm8gInByb2R1Y2UiIGNhZGEgdG9rZW4KICAgICAg'
        'ICBzZWxmLmFuY2hvcl9oZWFkID0gbm4uTGluZWFyKEQsIGNvbmZpZy5tYXhfZ3JhcGhfbm9kZXMp'
        'CgogICAgICAgICMgTWV0YSBoZWFkOiBjb25maWFuemEgeSBuZWNlc2lkYWQgZGUgY2xhcmlmaWNh'
        'Y2nDs24KICAgICAgICBzZWxmLm1ldGFfaGVhZCA9IE91dHB1dE1ldGFIZWFkKGNvbmZpZykKCiAg'
        'ICAgICAgIyDilIDilIAgV2VpZ2h0IHR5aW5nIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMgdG9rZW5fZW1iZWRkaW5nLndlaWdodCBbdm9j'
        'YWJfc2l6ZSwgRF0gPT0gbG1faGVhZC53ZWlnaHQgW3ZvY2FiX3NpemUsIERdCiAgICAgICAgIyBQ'
        'csOhY3RpY2EgZXN0w6FuZGFyIGVuIExNczogcmVkdWNlIHBhcsOhbWV0cm9zIHkgbWVqb3JhIGNv'
        'bnZlcmdlbmNpYS4KICAgICAgICBzZWxmLmxtX2hlYWQud2VpZ2h0ID0gc2VsZi50b2tlbl9lbWJl'
        'ZGRpbmcud2VpZ2h0CgogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygpCgogICAgZGVmIF9pbml0'
        'X3dlaWdodHMoc2VsZikgLT4gTm9uZToKICAgICAgICAiIiJJbmljaWFsaXphY2nDs24gZXN0w6Fu'
        'ZGFyIHBhcmEgZW1iZWRkaW5ncyB5IGhlYWRzLiIiIgogICAgICAgIG5uLmluaXQubm9ybWFsXyhz'
        'ZWxmLnRva2VuX2VtYmVkZGluZy53ZWlnaHQsIHN0ZD0wLjAyKQogICAgICAgIG5uLmluaXQubm9y'
        'bWFsXyhzZWxmLnBvc19lbWJlZGRpbmcud2VpZ2h0LCBzdGQ9MC4wMikKICAgICAgICAjIGxtX2hl'
        'YWQud2VpZ2h0IGVzIGVsIG1pc21vIHRlbnNvciBxdWUgdG9rZW5fZW1iZWRkaW5nLndlaWdodCAo'
        'dHlpbmcpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYuYW5jaG9yX2hlYWQud2VpZ2h0LCBz'
        'dGQ9MC4wMikKICAgICAgICBubi5pbml0Lnplcm9zXyhzZWxmLmFuY2hvcl9oZWFkLmJpYXMpCgog'
        'ICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICB0b2tlbl9pZHM6ICB0b3JjaC5U'
        'ZW5zb3IsICAgIyBbQiwgTF0KICAgICAgICBncmFwaF9yZXByOiB0b3JjaC5UZW5zb3IsICAgIyBb'
        'Qiwgbl9ub2Rlcywgbm9kZV9kaW1dCiAgICApIC0+IERlY29kZXJPdXRwdXQ6CiAgICAgICAgIiIi'
        'CiAgICAgICAgUGFzYSBwb3IgdG9kYXMgbGFzIGNhcGFzIHkgZ2VuZXJhIGxvZ2l0cyBwYXJhIGNh'
        'ZGEgcG9zaWNpw7NuLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB0b2tlbl9pZHM6ICBbQiwg'
        'TF0g4oCUIMOtbmRpY2VzIGRlIHRva2VucyBkZSBlbnRyYWRhICh0ZWFjaGVyLWZvcmNlZCkKICAg'
        'ICAgICAgICAgZ3JhcGhfcmVwcjogW0IsIG5fbm9kZXMsIG5vZGVfZGltXSDigJQgcmVwcmVzZW50'
        'YWNpw7NuIGRlbCBncmFmbyBjYXVzYWwKCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRGVj'
        'b2Rlck91dHB1dCBjb24gbG9naXRzIFtCLCBMLCB2b2NhYl9zaXplXSB5IG1ldGFkYXRvcwogICAg'
        'ICAgICIiIgogICAgICAgIEIsIEwgPSB0b2tlbl9pZHMuc2hhcGUKCiAgICAgICAgIyBQb3NpdGlv'
        'bnM6IFsxLCBMXSDigJQgYnJvYWRjYXN0IHNvYnJlIGVsIGJhdGNoCiAgICAgICAgcG9zID0gdG9y'
        'Y2guYXJhbmdlKEwsIGRldmljZT10b2tlbl9pZHMuZGV2aWNlLCBkdHlwZT10b3JjaC5sb25nKS51'
        'bnNxdWVlemUoMCkKCiAgICAgICAgIyBDb21iaW5hciB0b2tlbiBlbWJlZGRpbmcgKyBwb3NpdGlv'
        'bmFsIGVtYmVkZGluZwogICAgICAgIHggPSBzZWxmLnRva2VuX2VtYmVkZGluZyh0b2tlbl9pZHMp'
        'ICsgc2VsZi5wb3NfZW1iZWRkaW5nKHBvcykgICMgW0IsIEwsIERdCgogICAgICAgICMgUGFzYXIg'
        'cG9yIHRvZGFzIGxhcyBIeWJyaWREZWNvZGVyTGF5ZXJzCiAgICAgICAgZm9yIGxheWVyIGluIHNl'
        'bGYubGF5ZXJzOgogICAgICAgICAgICB4ID0gbGF5ZXIoeCwgZ3JhcGhfcmVwcikgICAjIFtCLCBM'
        'LCBEXQoKICAgICAgICAjIE5vcm1hbGl6YWNpw7NuIGZpbmFsCiAgICAgICAgeCA9IHNlbGYuZmlu'
        'YWxfbm9ybSh4KSAgICAgICAgICMgW0IsIEwsIERdCgogICAgICAgICMgSGVhZHMgZGUgc2FsaWRh'
        'CiAgICAgICAgbG9naXRzICAgICAgICA9IHNlbGYubG1faGVhZCh4KSAgICAgICAgIyBbQiwgTCwg'
        'dm9jYWJfc2l6ZV0KICAgICAgICBhbmNob3JfbG9naXRzID0gc2VsZi5hbmNob3JfaGVhZCh4KSAg'
        'ICAjIFtCLCBMLCBtYXhfZ3JhcGhfbm9kZXNdCiAgICAgICAgbWV0YSAgICAgICAgICA9IHNlbGYu'
        'bWV0YV9oZWFkKHgsIGdyYXBoX3JlcHIpCgogICAgICAgIHJldHVybiBEZWNvZGVyT3V0cHV0KAog'
        'ICAgICAgICAgICBsb2dpdHMgICAgICAgICAgICAgID0gbG9naXRzLAogICAgICAgICAgICBhbmNo'
        'b3JfbG9naXRzICAgICAgID0gYW5jaG9yX2xvZ2l0cywKICAgICAgICAgICAgY29uZmlkZW5jZSAg'
        'ICAgICAgICA9IG1ldGEuY29uZmlkZW5jZSwKICAgICAgICAgICAgbmVlZHNfY2xhcmlmaWNhdGlv'
        'biA9IG1ldGEubmVlZHNfY2xhcmlmaWNhdGlvbiwKICAgICAgICApCgogICAgIyDilIDilIAgVXRp'
        'bGlkYWRlcyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIAKCiAgICBkZWYgY291bnRfcGFyYW1ldGVycyhzZWxmKSAtPiBpbnQ6'
        'CiAgICAgICAgIiIiTsO6bWVybyB0b3RhbCBkZSBwYXLDoW1ldHJvcyBlbnRyZW5hYmxlcy4iIiIK'
        'ICAgICAgICAjIHRva2VuX2VtYmVkZGluZyB5IGxtX2hlYWQgY29tcGFydGVuIHBlc29zIOKGkiBj'
        'b250YXIgdW5hIHNvbGEgdmV6CiAgICAgICAgc2Vlbjogc2V0ID0gc2V0KCkKICAgICAgICB0b3Rh'
        'bCA9IDAKICAgICAgICBmb3IgcCBpbiBzZWxmLnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgaWYg'
        'aWQocCkgbm90IGluIHNlZW46CiAgICAgICAgICAgICAgICBzZWVuLmFkZChpZChwKSkKICAgICAg'
        'ICAgICAgICAgIGlmIHAucmVxdWlyZXNfZ3JhZDoKICAgICAgICAgICAgICAgICAgICB0b3RhbCAr'
        'PSBwLm51bWVsKCkKICAgICAgICByZXR1cm4gdG90YWwKCiAgICBkZWYgcGFyYW1ldGVyX2JyZWFr'
        'ZG93bihzZWxmKSAtPiBkaWN0OgogICAgICAgICIiIkRlc2dsb3NlIGRlIHBhcsOhbWV0cm9zIHBv'
        'ciBzdWItbcOzZHVsbyAoc2luIGRvYmxlLWNvbnRhciB3ZWlnaHQgdHlpbmcpLiIiIgogICAgICAg'
        'IGVtYmVkID0gc2VsZi50b2tlbl9lbWJlZGRpbmcud2VpZ2h0Lm51bWVsKCkKICAgICAgICBwb3Mg'
        'ICA9IHNlbGYucG9zX2VtYmVkZGluZy53ZWlnaHQubnVtZWwoKQogICAgICAgIGxheWVyc190b3Rh'
        'bCA9IHN1bSgKICAgICAgICAgICAgcC5udW1lbCgpIGZvciBsYXllciBpbiBzZWxmLmxheWVycyBm'
        'b3IgcCBpbiBsYXllci5wYXJhbWV0ZXJzKCkKICAgICAgICApCiAgICAgICAgcmV0dXJuIHsKICAg'
        'ICAgICAgICAgInRva2VuX2VtYmVkZGluZyI6ICAgZW1iZWQsCiAgICAgICAgICAgICJwb3NfZW1i'
        'ZWRkaW5nIjogICAgIHBvcywKICAgICAgICAgICAgImxheWVycyI6ICAgICAgICAgICAgbGF5ZXJz'
        'X3RvdGFsLAogICAgICAgICAgICAibG1faGVhZCI6ICAgICAgICAgICAwLCAgIyB0aWVkIGNvbiB0'
        'b2tlbl9lbWJlZGRpbmcKICAgICAgICAgICAgImFuY2hvcl9oZWFkIjogICAgICAgc3VtKHAubnVt'
        'ZWwoKSBmb3IgcCBpbiBzZWxmLmFuY2hvcl9oZWFkLnBhcmFtZXRlcnMoKSksCiAgICAgICAgICAg'
        'ICJtZXRhX2hlYWQiOiAgICAgICAgIHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5tZXRhX2hl'
        'YWQucGFyYW1ldGVycygpKSwKICAgICAgICAgICAgInRvdGFsX3VuaXF1ZSI6ICAgICAgc2VsZi5j'
        'b3VudF9wYXJhbWV0ZXJzKCksCiAgICAgICAgfQo='
    ),
    'router/__init__.py': (
        'IiIiCkFJT04tQyByb3V0ZXIg4oCUIENPUkFQaXBlbGluZSBlbmQtdG8tZW5kLgoKQ29uZWN0YSBT'
        'RSDihpIgR0Mg4oaSIENSRSDihpIgU0QgZW4gdW4gbcOzZHVsbyDDum5pY28gZGlmZXJlbmNpYWJs'
        'ZS4KIiIiCgpmcm9tIC5waXBlbGluZSBpbXBvcnQgQ09SQUNvbmZpZywgQ09SQVBpcGVsaW5lLCBQ'
        'aXBlbGluZU91dHB1dAoKX19hbGxfXyA9IFsKICAgICJDT1JBQ29uZmlnIiwKICAgICJDT1JBUGlw'
        'ZWxpbmUiLAogICAgIlBpcGVsaW5lT3V0cHV0IiwKXQo='
    ),
    'router/pipeline.py': (
        '...'
        'ICAgICAgfQo='
    ),
    'experiments/__init__.py': (
        'IyBleHBlcmltZW50cyBwYWNrYWdlCg=='
    ),
    'experiments/train_cora_50m.py': (
        '...'
        'X18iOgogICAgbWFpbigpCg=='
    ),
}

for rel_path, b64_content in files_b64.items():
    if isinstance(b64_content, tuple):
        b64_content = ''.join(b64_content)
    content = base64.b64decode(b64_content).decode('utf-8')
    (ROOT / rel_path).write_text(content, encoding='utf-8')
    print(f'  ✓ {rel_path}')

sys.path.insert(0, str(ROOT))
print(f'\n✓ {len(files_b64)} archivos escritos en {ROOT}')
print('✓ sys.path configurado')


In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free_b, total_b = torch.cuda.mem_get_info()
    print(f'GPU     : {name}')
    print(f'VRAM    : {free_b/1e9:.1f} GB libre / {total_b/1e9:.1f} GB total')
    for n_params, desc in [(37_053_274, '256D-8L (37M)'), (5_100_000, '128D-4L (5M)')]:
        model_mb  = n_params * 4 / 1e6
        optim_mb  = n_params * 4 * 2 / 1e6
        total_est = model_mb + optim_mb + 300
        ok = 'OK' if total_est/1024 < free_b/1e9 else 'RIESGO'
        print(f'  {desc}: ~{total_est:.0f} MB estimado [{ok}]')
else:
    print('Sin GPU — ve a Runtime -> Change runtime type -> T4 GPU')

In [ ]:
import sys, os, time, torch, shutil, glob
sys.path.insert(0, '/content/aion_c')

import experiments.train_cora_50m as trainer
from router.pipeline import CORAConfig

# ── Constantes fast ──────────────────────────────────────────────────────────
N_STEPS      = 2000
CRE_ITERS    = 3        # 3 en vez de 10 — principal aceleración
ACCUM_STEPS  = 1        # sin gradient accumulation
EVAL_EVERY   = 400
CKPT_EVERY   = 1000
PRINT_EVERY  = 50
TARGET_MIN   = 25.0     # abortar config si estimado > 25 min
DRIVE_DIR    = '/content/drive/MyDrive/cora_50m_fast'

# Aplicar constantes al módulo
trainer.N_DATASET   = 5000
trainer.N_STEPS     = N_STEPS
trainer.CRE_ITERS   = CRE_ITERS
trainer.ACCUM_STEPS = ACCUM_STEPS
trainer.EVAL_EVERY  = EVAL_EVERY
trainer.CKPT_EVERY  = CKPT_EVERY
trainer.PRINT_EVERY = PRINT_EVERY
trainer.TRAIN_FRAC  = 0.80

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Dataset + vocab (una sola vez, se reutilizan) ────────────────────────────
print('\n[setup] Generando dataset...')
train_ex, eval_ex = trainer.build_dataset(5000)
vocab = trainer.build_vocab(train_ex + eval_ex, max_size=8000)
actual_vocab = len(vocab)
print(f'[setup] vocab={actual_vocab}  train={len(train_ex)}  eval={len(eval_ex)}')

# Cachear para que train() no regenere
_t, _e, _v = train_ex, eval_ex, vocab
trainer.build_dataset = lambda n, seed=42: (_t, _e)
trainer.build_vocab   = lambda examples, max_size: _v


# ── Factory de configs candidatos ────────────────────────────────────────────
def make_cfg(hidden_dim, n_layers, vocab_size):
    n_heads = 8 if hidden_dim >= 256 else 4
    return CORAConfig(
        hidden_dim   = hidden_dim,
        vocab_size   = vocab_size,
        enc_n_layers  = n_layers,
        enc_state_dim = 16,
        enc_expand    = 2,
        enc_d_conv    = 4,
        enc_ffn_mult  = 4,
        cryst_max_nodes      = 32,
        cryst_n_heads        = n_heads,
        cryst_node_threshold = 0.01,
        cryst_edge_threshold = 0.01,
        cre_edge_dim         = 64,
        cre_message_dim      = min(128, hidden_dim // 2),
        cre_n_message_layers = 2,
        cre_max_iterations   = CRE_ITERS,
        pad_n_slots  = 32,
        pad_slot_dim = min(128, hidden_dim // 2),
        dec_n_layers    = n_layers,
        dec_n_heads     = n_heads,
        dec_max_seq_len = 256,
        dec_state_dim   = 16,
        dec_expand      = 2,
        dec_d_conv      = 4,
        dec_ffn_mult    = 4,
    )


# ── Benchmark de 1 step ───────────────────────────────────────────────────────
def benchmark_ms(cfg, device, vocab, train_ex, n_cre, n_warmup=3):
    from encoder import StreamEncoder
    from crystallizer import GraphCrystallizer
    from cre import CausalReasoningEngine, DifferentiableScratchPad
    from decoder import StreamDecoder

    enc  = StreamEncoder(cfg.encoder_config()).to(device)
    crys = GraphCrystallizer(cfg.crystallizer_config()).to(device)
    cre  = CausalReasoningEngine(cfg.cre_config()).to(device)
    pad  = DifferentiableScratchPad(cfg.scratch_pad_config()).to(device)
    dec  = StreamDecoder(cfg.decoder_config()).to(device)
    all_p = (list(enc.parameters()) + list(crys.parameters()) +
             list(cre.parameters()) + list(pad.parameters()) + list(dec.parameters()))
    n_p = sum(p.numel() for p in all_p if id(p))
    opt = torch.optim.AdamW(all_p, lr=3e-4)

    ex    = train_ex[0]
    q_len = min(len(ex.problem_text.lower().split()), 80)
    q_ids = vocab.to_tensor(vocab.encode(ex.problem_text, 80), device, 80)
    a_ids = vocab.to_tensor(vocab.encode(ex.answer, 48, add_eos=True), device)

    # Warmup
    for _ in range(n_warmup):
        opt.zero_grad()
        co, gr, cf, nv, lm, enc_out = trainer.forward_full(enc,crys,cre,pad,dec,q_ids,a_ids,cfg,n_cre,vocab)
        tot, _ = trainer.compute_all_losses(co,cf,nv,lm,ex.graph,a_ids,cfg,vocab,device,enc_out,ex.entity_spans,q_len)
        tot.backward()
    if device.type == 'cuda': torch.cuda.synchronize()

    opt.zero_grad()
    t0 = time.perf_counter()
    co, gr, cf, nv, lm, enc_out = trainer.forward_full(enc,crys,cre,pad,dec,q_ids,a_ids,cfg,n_cre,vocab)
    tot, _ = trainer.compute_all_losses(co,cf,nv,lm,ex.graph,a_ids,cfg,vocab,device,enc_out,ex.entity_spans,q_len)
    tot.backward()
    if device.type == 'cuda': torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000

    del enc,crys,cre,pad,dec,opt,all_p
    if device.type=='cuda': torch.cuda.empty_cache()
    return ms, n_p


# ── Auto-selección de config ──────────────────────────────────────────────────
CANDIDATES = [
    (256, 8, 'hidden_dim=256  layers=8  (~37M params)'),
    (128, 4, 'hidden_dim=128  layers=4  (~ 5M params)'),
]

selected_cfg  = None
selected_desc = None
selected_ms   = None

print('\n[timing] Midiendo throughput por configuracion...')
print('-' * 60)

for hidden_dim, n_layers, desc in CANDIDATES:
    cfg_try = make_cfg(hidden_dim, n_layers, actual_vocab)
    ms, n_p = benchmark_ms(cfg_try, device, vocab, train_ex, CRE_ITERS)
    est_min = ms * N_STEPS / 60_000
    ok = est_min <= TARGET_MIN
    status = 'OK' if ok else 'lento'
    print(f'  {desc}')
    print(f'    {ms:6.0f} ms/step  ->  {N_STEPS} steps = {est_min:.1f} min  [{status}]')

    if ok:
        selected_cfg  = cfg_try
        selected_desc = desc
        selected_ms   = ms
        print(f'    -> SELECCIONADA')
        break
    else:
        print(f'    -> probando siguiente...')

if selected_cfg is None:
    selected_cfg  = make_cfg(128, 4, actual_vocab)
    selected_desc = 'hidden_dim=128  layers=4  (fallback minimo)'
    ms, _ = benchmark_ms(selected_cfg, device, vocab, train_ex, CRE_ITERS, n_warmup=1)
    selected_ms = ms

print('-' * 60)
print(f'\nCONFIG FINAL : {selected_desc}')
print(f'TIEMPO ESTIM : {selected_ms * N_STEPS / 60_000:.1f} min ({selected_ms:.0f} ms/step x {N_STEPS} steps)')
print()

# ── Patch make_50m_config para devolver la config seleccionada ───────────────
_frozen = selected_cfg
trainer.make_50m_config = lambda vocab_size=None: _frozen

# ── Patch save_checkpoint para guardar en Drive ──────────────────────────────
_orig_save = trainer.save_checkpoint
def _drive_save(step, encoder, crystallizer, cre, scratch_pad, decoder,
                optimizer, scheduler, loss_history, ckpt_dir):
    path = _orig_save(step, encoder, crystallizer, cre, scratch_pad, decoder,
                      optimizer, scheduler, loss_history,
                      f'{DRIVE_DIR}/checkpoints')
    print(f'  [Drive] checkpoint guardado: {os.path.basename(path)}')
    return path
trainer.save_checkpoint = _drive_save

# ── Lanzar entrenamiento ──────────────────────────────────────────────────────
trainer.train()

# ── Copiar resultados a Drive ─────────────────────────────────────────────────
result_files = sorted(glob.glob('/content/aion_c/experiments/results/train_cora_50m_*.json'))
for src_f in result_files:
    dst = f'{DRIVE_DIR}/results/{os.path.basename(src_f)}'
    shutil.copy(src_f, dst)
    print(f'[Drive] resultados: {dst}')

In [ ]:
import json, glob, os
import matplotlib.pyplot as plt

DRIVE_DIR = '/content/drive/MyDrive/cora_50m_fast'

# Buscar resultados: primero local, luego en Drive
result_files = (sorted(glob.glob('/content/aion_c/experiments/results/train_cora_50m_*.json')) +
                sorted(glob.glob(f'{DRIVE_DIR}/results/train_cora_50m_*.json')))

if not result_files:
    print('No se encontraron archivos de resultados.')
    print('Asegurate de haber ejecutado la celda de entrenamiento.')
else:
    with open(result_files[-1], encoding='utf-8') as f:
        data = json.load(f)

    cfg_used = data.get('config', {})
    summary  = data.get('summary', {})
    print(f'Modelo   : hidden_dim={cfg_used.get("hidden_dim")}  params={cfg_used.get("n_params",0):,}')
    print(f'Steps    : {cfg_used.get("n_steps")}  LM mejora: {summary.get("lm_loss_improvement_pct",0):.1f}%')
    print(f'Tiempo   : {summary.get("total_seconds",0)/60:.1f} min  ({summary.get("avg_ms_per_step",0):.0f} ms/step)')
    print()

    curve  = data['loss_curve']
    steps  = [r['step']  for r in curve]
    total  = [r['total'] for r in curve]
    lm_pts = [(r['step'], r['lm']) for r in curve if r.get('lm')]

    # Rolling average helper
    def smooth(vals, w=30):
        out = []
        for i in range(len(vals)):
            sl = vals[max(0, i-w):i+1]
            out.append(sum(sl)/len(sl))
        return out

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(steps, total, alpha=0.25, color='steelblue', linewidth=0.7)
    axes[0].plot(steps, smooth(total), color='steelblue', linewidth=2, label='MA-30')
    axes[0].set_title('Loss Total (NC + Rel + Coh + LM)')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    if lm_pts:
        lm_s, lm_v = zip(*lm_pts)
        axes[1].plot(lm_s, lm_v, alpha=0.25, color='coral', linewidth=0.7)
        axes[1].plot(lm_s, smooth(list(lm_v)), color='coral', linewidth=2)
        axes[1].set_title('LM Loss (generacion de respuestas)')
        axes[1].set_xlabel('Step'); axes[1].grid(alpha=0.3)

    plt.suptitle('CORA 50M Fast — Curva de entrenamiento', fontsize=14, fontweight='bold')
    plt.tight_layout()
    out_png = f'{DRIVE_DIR}/loss_curve_fast.png'
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Grafico guardado: {out_png}')

    # Eval snapshots
    snaps = data.get('eval_snapshots', [])
    if snaps:
        print()
        print('=' * 72)
        print('EVALUACIONES DURANTE ENTRENAMIENTO')
        print('=' * 72)
        print(f'  {"Step":>6}  {"NodeAcc":>8}  {"RelRecall":>10}  {"WordF1":>8}')
        print('  ' + '-' * 40)
        for s in snaps:
            print(f'  {s["step"]:>6}  {s["avg_node_acc"]:>8.1%}  '
                  f'{s["avg_rel_recall"]:>10.1%}  {s["avg_word_f1"]:>8.1%}')

        # Ultimos 5 ejemplos del ultimo snapshot
        last = snaps[-1]
        print()
        print('=' * 72)
        print(f'ULTIMOS 5 EJEMPLOS EVAL (step {last["step"]})')
        print('=' * 72)
        for i, ex in enumerate(last.get('examples', [])[:5], 1):
            print(f'\n[{i}] Nivel {ex["level"]}  '
                  f'Nodos GT:{ex["gt_n"]} Pred:{ex["pred_n"]}  '
                  f'NodeAcc:{ex["node_acc"]:.0%}  RelRecall:{ex["rel_recall"]:.0%}')
            print(f'  Input: {ex["q_preview"]}...')
            print(f'  GT   : {ex["gt_answer"][:80]}')
            gen = ex.get("gen_answer", "") or "(vacio)"
            print(f'  Gen  : {gen[:80]}')
            print(f'  WordF1: {ex["word_overlap"]:.0%}')